# 02 — Cleaning and features

Second of two notebooks. Notebook `01` loaded the raw M1 meter readings and the Open-Meteo
weather for the site and wrote them into `notebooks/data/`. This one is the development
itself: we restrict the data to the fifteen meters this study covers, turn their 15-minute
readings into a clean hourly series, define every feature the forecasting models consume,
and then measure — on a holdout no model ever sees — which of those features actually earn
their place.

Every step described here is also implemented as a function in `celine.forecasting`, so the
command-line pipeline runs exactly the same code on exactly the same thresholds. Where a
step is a library call we call it; where it is a decision we make it here and say why.

## What is in here

1. Setup and inputs — the cohort and the configuration
2. Cleaning, step by step — hourly aggregation, the DST nights, the noise floor, the
   regular grid and its interpolation, the weather merge, calendar features, outlier flags
3. A per-device data-quality table
4. The features — the forecasting setup, the observability rule, the catalogue, the code,
   a leak test and a look at the matrix
5. Which features are useful? — 26 models, a 7-origin holdout, permutation importance and
   a verdict per feature
6. Saved outputs
7. Findings

## How to run

From the repository root, with the `notebooks` extra installed:

```
uv run jupyter nbconvert --to notebook --execute --inplace \
    --ExecutePreprocessor.timeout=3600 notebooks/02_cleaning_and_features.ipynb
```

It needs no database and no network: the two parquet extracts written by notebook `01` are
the only inputs. On a 16-core machine the whole notebook runs in about six minutes,
most of it in the 52 LightGBM fits of section 5 plus the twelve of its
early-stopping check.

## Units — read this before any number below

* The raw meter columns are **energy in kWh per 15-minute interval**, not power.
  The hourly value is therefore the **SUM** of the (up to four) quarters, never the mean.
* After hourly aggregation every energy quantity is **kWh per hour**.
* The meters sit at the grid connection point, so `grid_import` is energy drawn *from*
  the grid and `grid_export` is energy pushed *to* it — not household consumption and PV
  generation as such.
* Timestamps are **tz-aware UTC** everywhere; only the calendar features are computed on
  the local (Europe/Rome) wall clock.

## 1. Setup and inputs

Imports and the two parquet extracts. `DATA_DIR` is `data/` **next to this notebook**
(`notebooks/data/`), which the repository's ignore rules already cover — nothing read or
written here can be committed by accident. No path is hardcoded: the repository root is
found by walking up to the directory holding `pyproject.toml`.

In [ ]:
import json
import time
import warnings
from pathlib import Path

import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
# numpy warns on every all-NaN mean/median, which is the intended value for a lag that
# reaches back before the device existed.
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 80)

NB_START = time.time()

# The notebook can be started from its own folder or from the repository root.
NB_DIR = Path.cwd().resolve()
if not (NB_DIR / "data").exists() and (NB_DIR / "notebooks" / "data").exists():
    NB_DIR = NB_DIR / "notebooks"

# Walk up from the notebook folder until we find the folder that holds pyproject.toml.
REPO_ROOT = None
for folder in [NB_DIR] + list(NB_DIR.parents):
    if (folder / "pyproject.toml").exists():
        REPO_ROOT = folder
        break
if REPO_ROOT is None:
    raise FileNotFoundError("could not find pyproject.toml above the notebook folder")

DATA_DIR = NB_DIR / "data"

# The site the meters sit on (Folgaria, Trentino), same constants as notebook 01.
SITE_LAT, SITE_LON = 45.9167, 11.1667
LOCAL_TZ = "Europe/Rome"
print("repository root:", REPO_ROOT)
print("data directory:", DATA_DIR)
print(f"site: lat {SITE_LAT}, lon {SITE_LON}, tz {LOCAL_TZ}")

### The cohort

These fifteen meters are the ones that matter most to the project, so we do the whole
development on them. They are named explicitly below and the restriction is applied to the
raw extract immediately, so every number in this notebook — cleaning statistics, the
quality table, the feature matrices, the models — describes these fifteen and nothing else.

One caveat about their history. Five of them (`...89CF4`, `...89ED4`, `...8A0D0`,
`...8AA78`, `...2E6EC`) appear in the CELINE tables only from 2026-05-21, but their earlier
readings do exist, in a separate source: the `mySET` table, which notebook 01 will read in
a later revision. The four-month span those five carry in this run is therefore a property
of the extract, not of the meters; once that history is in, all fifteen have a long
training record.

Two inputs are read:

* **`raw_meters_m1.parquet`** — the M1 grid-exchange readings at 15-minute resolution;
* **`weather_openmeteo.parquet`** — hourly Open-Meteo ICON-D2 weather for the site,
  covering the whole meter span with no gaps and no nulls.

In [ ]:
# The fifteen meters this study covers.
DEVICES = [
    "c2g-3987B3594", "c2g-3987C9968", "c2g-57CFA0F18", "c2g-57CFAAA3C", "c2g-57CFB17D0",
    "c2g-57CFBC3F0", "c2g-686DEF19C", "c2g-793E62FD4", "c2g-57CFCD3D0", "c2g-57CFB19D8",
    "c2g-9FFB89CF4", "c2g-9FFB89ED4", "c2g-9FFB8A0D0", "c2g-9FFB8AA78", "c2g-DD6C2E6EC",
]

paths = {
    "raw_meters_m1": DATA_DIR / "raw_meters_m1.parquet",
    "weather_openmeteo": DATA_DIR / "weather_openmeteo.parquet",
}
for name, path in paths.items():
    if not path.exists():
        raise FileNotFoundError(
            f"missing extract {name}: run notebooks/01_data_loading_and_eda.ipynb first"
        )

df_extract = pd.read_parquet(paths["raw_meters_m1"])
weather = pd.read_parquet(paths["weather_openmeteo"])

# Restrict to the cohort right away: everything downstream sees only these fifteen.
missing_devices = []
for device_id in DEVICES:
    if device_id not in set(df_extract["device_id"]):
        missing_devices.append(device_id)
if missing_devices:
    raise ValueError(f"cohort devices absent from the extract: {missing_devices}")

df_meters = df_extract[df_extract["device_id"].isin(DEVICES)].reset_index(drop=True)

print(f"extract: {len(df_extract):,} readings, {df_extract['device_id'].nunique()} devices")
print(f"cohort:  {len(df_meters):,} readings, {df_meters['device_id'].nunique()} devices "
      f"({len(df_meters) / len(df_extract) * 100:.1f}% of the extract)")
print("cohort span (UTC):", df_meters["ts"].min(), "->", df_meters["ts"].max())
print(f"weather: {len(weather):,} hourly rows,", weather["datetime"].min(), "->",
      weather["datetime"].max(), f"| nulls: {int(weather.isna().sum().sum())}")
df_meters.head(3)

In [ ]:
# One row per meter: when it came online, when it stopped, how much it moves and in which
# counting frame. Values are kWh per 15-minute interval, as they come off the meter.
cohort = df_meters.groupby("device_id").agg(
    readings=("ts", "size"),
    first_reading=("ts", "min"),
    last_reading=("ts", "max"),
    mean_import_kwh_15min=("consumption_kwh", "mean"),
    mean_export_kwh_15min=("production_kwh", "mean"),
)

# Each meter uses a single counting frame for its whole history, so one label per device.
frame_labels = {}
for device_id, device_rows in df_meters.groupby("device_id"):
    frame_labels[device_id] = "+".join(sorted(device_rows["cf_type"].unique()))
cohort["frame"] = pd.Series(frame_labels)

span_seconds = (cohort["last_reading"] - cohort["first_reading"]).dt.total_seconds()
cohort["span_days"] = (span_seconds / 86400).round(1)
cohort["device"] = cohort.index.str[-5:]
cohort = cohort.sort_values(["first_reading", "device_id"])

extract_end = df_meters["ts"].max()
still_running = cohort["last_reading"] > extract_end - pd.Timedelta(days=7)
never_exports = cohort["mean_export_kwh_15min"] == 0

print("onboarding waves (first reading, UTC date):")
print(cohort["first_reading"].dt.date.value_counts().sort_index().rename("devices").to_frame())
print(f"\nstill reporting within 7 days of the extract end: {int(still_running.sum())} of "
      f"{len(cohort)}")
print("stopped early:", list(cohort.loc[~still_running, "device"]))
print("never export a single kWh:", list(cohort.loc[never_exports, "device"]))
print(f"span: min {cohort['span_days'].min():.0f} d, median "
      f"{cohort['span_days'].median():.0f} d, max {cohort['span_days'].max():.0f} d")

cohort_view = cohort.set_index("device")
cohort_view[["frame", "readings", "first_reading", "last_reading", "span_days",
             "mean_import_kwh_15min", "mean_export_kwh_15min"]].round(4)

### The configuration

Every threshold used below comes from the packaged YAML
(`core/config_data/default_config.yaml`) through `load_config()`. Printing it here means no
number in this notebook is a magic constant: if a verdict changes, it is because the
configuration changed.

In [ ]:
from celine.forecasting.core.config import load_config

cfg = load_config()

rows = []
for key, value in cfg.cleaning.items():
    rows.append({"section": "cleaning", "key": key, "value": str(value)})

# Only the one sufficiency threshold this notebook uses: the PV flag of section 3.
rows.append({"section": "sufficiency", "key": "export_min_mean_kwh",
             "value": str(cfg.sufficiency["export_min_mean_kwh"])})

for key, value in cfg.lgb_params.items():
    rows.append({"section": "lgb_params", "key": key, "value": str(value)})
for target, overrides in cfg.raw["lgb_params_by_target"].items():
    rows.append({"section": "lgb_params_by_target", "key": target, "value": str(overrides)})

# The training controls section 5 reuses.
for name in ("num_boost_round", "early_stopping_rounds", "validation_split_fraction"):
    rows.append({"section": "training", "key": name, "value": str(cfg.raw[name])})

for name in ("local_tz", "targets", "forecast_horizon", "random_seed"):
    rows.append({"section": "top-level", "key": name, "value": str(getattr(cfg, name))})

pd.DataFrame(rows)

## 2. Cleaning, step by step

Six transformations turn the raw 15-minute readings into the hourly frame the models read.
Running them as one call hides what each one does, so below each step is called on its own
and measured; section 2.7 then runs the chain end to end and checks the two agree.

### 2.1 Hourly aggregation — 15-minute energy to hourly energy

Three things happen, in this order:

1. rows before `cleaning.start_date` are dropped (here `start_date` is null, so nothing is);
2. `(device_id, ts)` duplicates are dropped with `keep="last"`;
3. the quarters are **summed** into their hour, and hours with fewer than four quarters are
   **scaled by `4 / n_quarters`** instead of being discarded.

Step 3 is the one that deserves attention. Scaling is the right call for energy — a partial
hour would otherwise read as a dip that never happened — but it is an extrapolation: an
hour with a single quarter is multiplied by four. The `n_quarters` count is not kept, so we
recompute it from the raw frame to see how often that happens and how aggressive the
scaling gets.

In [ ]:
from celine.forecasting.core import cleaning as clean

print(f"raw cohort rows: {len(df_meters):,}")
print("raw (device_id, ts) duplicates:",
      int(df_meters.duplicated(subset=["device_id", "ts"]).sum()))

hourly = clean.aggregate_to_hourly(df_meters, cfg)
print(f"hourly rows: {len(hourly):,}")
print(f"ratio raw/hourly: {len(df_meters) / len(hourly):.2f} "
      f"(a perfect 4 would mean no missing quarter anywhere)")
print("columns:", list(hourly.columns))
print("devices:", hourly["device_id"].nunique())

# n_quarters recomputed from the raw frame, since the aggregation drops it.
dedup = df_meters.drop_duplicates(subset=["device_id", "ts"], keep="last")
dedup = dedup.assign(ts_hour=dedup["ts"].dt.floor("h"))
nq = dedup.groupby(["device_id", "ts_hour"]).size().rename("n_quarters")

is_partial_hour = nq < 4
partial_hours = int(is_partial_hour.sum())
print(f"\npartial hours (n_quarters < 4): {partial_hours:,} "
      f"({is_partial_hour.mean() * 100:.2f}% of hours)")
print("agrees with the partial_hour flag:",
      int(hourly["partial_hour"].sum()) == partial_hours)

distribution = nq.value_counts().sort_index().rename("hours").to_frame()
distribution["scale_applied"] = (4 / distribution.index).round(2)
distribution["share_pct"] = (distribution["hours"] / len(nq) * 100).round(3)
print("\nn_quarters distribution:")
print(distribution)

In [ ]:
counts = nq.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))

# Left: all hours on a log scale, because full hours dwarf the partial ones.
ax = axes[0]
ax.bar(counts.index, counts.values, color="tab:blue")
for x, v in zip(counts.index, counts.values):
    ax.text(x, v * 1.12, f"{v:,}", ha="center")
ax.set_yscale("log")
ax.set_xticks([1, 2, 3, 4])
ax.set_ylim(top=counts.max() * 4)
ax.set_title("Quarters per aggregated hour")
ax.set_xlabel("quarter-hour readings in the hour")
ax.set_ylabel("hours (log scale)")

# Right: the partial hours only, on a linear scale.
ax = axes[1]
partial = counts[counts.index < 4]
ax.bar(partial.index, partial.values, color="tab:orange")
for x, v in zip(partial.index, partial.values):
    ax.text(x, v, f"{v:,}", ha="center", va="bottom")
ax.set_xticks([1, 2, 3])
ax.set_ylim(0, partial.max() * 1.2)
ax.set_title("Partial hours only")
ax.set_xlabel("quarter-hour readings in the hour")
ax.set_ylabel("hours")
plt.tight_layout()
plt.show()
print("Hours with n < 4 are scaled by 4/n, so n = 1 means one reading multiplied by four.")

#### The DST fall-back night

Notebook 01 found duplicated `(device_id, ts)` pairs in the source table on the night of
**2025-10-26**, when the local wall clock runs 02:00–02:59 twice. The loader dropped them,
so the extract read here has none left. What is left is the hole those dropped rows leave
on the UTC clock, which we look at directly.

In [ ]:
# The UTC window that covers the local DST fall-back night of 2025-10-26.
night_start = pd.Timestamp("2025-10-25 20:00", tz="UTC")
night_end = pd.Timestamp("2025-10-26 05:00", tz="UTC")
night = df_meters[(df_meters["ts"] >= night_start) & (df_meters["ts"] < night_end)]

# Readings and distinct devices per UTC hour of that night.
night_hours = night.assign(h=night["ts"].dt.floor("h"))
per_hour = night_hours.groupby("h").agg(readings=("ts", "size"),
                                        devices=("device_id", "nunique"))

# Reindex onto every hour of the window so an empty hour shows up as a zero row.
all_hours = pd.date_range(per_hour.index.min(), per_hour.index.max(), freq="h")
per_hour = per_hour.reindex(all_hours, fill_value=0)
per_hour["ts_local"] = per_hour.index.tz_convert(LOCAL_TZ).strftime("%H:%M %Z")

# How many of the fifteen were online that night, computed from the data.
devices_online = night["device_id"].nunique()
print("devices online that night:", devices_online,
      f"(a full hour is {devices_online} x 4 = {devices_online * 4} readings)")
print(per_hour)

missing_hours = per_hour.index[per_hour["readings"] == 0]
print("\nUTC hours with no reading at all:", list(missing_hours))
print("The duplicated local hour collapses onto a single UTC hour, so exactly one UTC hour")
print("goes missing. It is a one-hour hole, which the regular grid of 2.3 interpolates.")

#### The DST spring-forward night

The opposite event is **2026-03-29**, when the local clock jumps from 02:00 straight to
03:00 and the wall-clock hour 02:00–02:59 never happens. UTC has no such jump, so the meter
series should show a perfectly ordinary night. We check it rather than assume it: a missing
UTC hour there would poison every lag that lands on it, exactly like the fall-back hole.

In [ ]:
# The UTC window that covers the local spring-forward night of 2026-03-29.
spring_start = pd.Timestamp("2026-03-28 20:00", tz="UTC")
spring_end = pd.Timestamp("2026-03-29 06:00", tz="UTC")
spring = df_meters[(df_meters["ts"] >= spring_start) & (df_meters["ts"] < spring_end)]

spring_hours = spring.assign(h=spring["ts"].dt.floor("h"))
spring_per_hour = spring_hours.groupby("h").agg(readings=("ts", "size"),
                                                devices=("device_id", "nunique"))
spring_all_hours = pd.date_range(spring_start, spring_end - pd.Timedelta(hours=1), freq="h")
spring_per_hour = spring_per_hour.reindex(spring_all_hours, fill_value=0)
spring_per_hour["ts_local"] = spring_per_hour.index.tz_convert(LOCAL_TZ).strftime("%H:%M %Z")

spring_online = spring["device_id"].nunique()
print("devices online that night:", spring_online)
print(spring_per_hour)

spring_missing = spring_per_hour.index[spring_per_hour["readings"] == 0]
print("\nUTC hours with no reading at all:", list(spring_missing))

# The local clock really does skip 02:00, which is why the check is worth making.
local_labels = sorted(set(spring_per_hour["ts_local"].str[:5]))
print("local hours present in the window:", local_labels)

# The same check on the weather frame, which the merge of 2.4 joins on this hour.
weather_in_window = weather[(weather["datetime"] >= spring_start)
                            & (weather["datetime"] < spring_end)]
print("weather rows in the same window:", len(weather_in_window), "of",
      len(spring_all_hours), "hours")
assert len(spring_missing) == 0, "the spring-forward night lost a UTC hour"
assert len(weather_in_window) == len(spring_all_hours), "weather lost a spring-forward hour"
print("\nNo hole: UTC runs straight through the jump, so the spring-forward night costs")
print("nothing. Only the autumn fall-back leaves a gap to repair.")

### 2.2 Grid import / export and the noise floor

`grid_import = max(M1_cons, 0)`, `grid_export = max(M1_prod, 0)`,
`net_exchange = grid_export - grid_import`, and anything below `cleaning.noise_floor_kwh`
(0.020 kWh/h) is snapped to exactly zero.

The floor exists because a meter that is idle does not report a clean zero: it reports
milliwatt-hours of measurement noise. Leaving them in would make "is this device exporting
right now?" a question about noise, and it would blur the zero-inflated `grid_import`
target. The cost is that genuinely tiny flows are erased, so we quantify it.

In [ ]:
derived = clean.add_derived_metrics(hourly, cfg)
floor = float(cfg.cleaning["noise_floor_kwh"])

# What the two targets look like before the noise floor is applied.
before = pd.DataFrame({
    "grid_import": derived["M1_cons"].clip(lower=0),
    "grid_export": derived["M1_prod"].clip(lower=0),
})

# One row per target: how many values the floor snapped to zero, and the energy that cost.
rows = []
for col in ("grid_import", "grid_export"):
    b = before[col]
    a = derived[col]
    is_below_floor = (b > 0) & (b < floor)
    zeroed = int(is_below_floor.sum())
    rows.append({
        "target": col,
        "rows": len(a),
        "zero_share_before_%": round((b == 0).mean() * 100, 2),
        "zero_share_after_%": round((a == 0).mean() * 100, 2),
        "values_zeroed_by_floor": zeroed,
        "share_zeroed_%": round(zeroed / len(a) * 100, 3),
        "energy_erased_kWh": round(float(b[is_below_floor].sum()), 2),
        "total_energy_kWh": round(float(b.sum()), 1),
    })

print("noise floor:", floor, "kWh/h")
print("negative raw values clipped: M1_cons", int((derived["M1_cons"] < 0).sum()),
      ", M1_prod", int((derived["M1_prod"] < 0).sum()))
pd.DataFrame(rows)

### 2.3 A regular hourly grid, and why we interpolate

**Why.** The lag features of section 4 look up fixed offsets: the same hour one day ago,
two days ago, seven days ago, and a 24-hour rolling window ending at the forecast origin.
Those look-ups are done by timestamp, so a missing hour does not shift the series — it
turns the lag into NaN. One absent hour therefore knocks a hole in `same_hour_1d` a day
later, in `same_hour_7d` a week later and in `same_hour_28d` four weeks later: a single
transmission hiccup costs feature values for **fourteen days** of downstream rows. That is
out of all proportion to the information actually lost, because a one-hour hole is almost
always a transmission hiccup rather than a real outage — the DST hole of 2.1 is exactly
one.

**How.** Every device is reindexed onto a continuous hourly range, and per device, per
numeric column, pandas fills with `interpolate(method="linear", limit=max_gap_hours)` and
clips the result at zero. `cleaning.max_gap_hours` is **1**, so only single missing hours
are filled; anything longer stays NaN and `gap_flag` marks it. A linear fill over one hour
is bounded by the hour-to-hour variability of the series, which we measure below rather
than assert.

The reindex is done on the **global** min→max range over all devices, so a device onboarded
in May 2026 gets nine months of empty rows in front of its first reading. Those rows are
not missing data — they are hours in which the device did not exist. We count them
separately and then drop them.

In [ ]:
grid = clean.build_regular_grid(derived, cfg)

full_range = pd.date_range(derived["ts_hour"].min(), derived["ts_hour"].max(), freq="h")
print("global hourly range:", full_range[0], "->", full_range[-1],
      f"({len(full_range):,} hours)")
print(f"rows before: {len(derived):,}")
print(f"rows after: {len(grid):,} "
      f"(= {grid['device_id'].nunique()} devices x {len(full_range):,} hours)")

# Mark the rows that already existed before the reindex, so the created ones stand out.
observed = derived[["device_id", "ts_hour"]].copy()
observed["was_observed"] = True
grid = grid.merge(observed, on=["device_id", "ts_hour"], how="left")
grid["was_observed"] = grid["was_observed"].fillna(False).astype(bool)

# A created row that carries a value was interpolated; one still NaN sat in too long a gap.
was_created = ~grid["was_observed"]
interp = was_created & grid["M1_cons"].notna()
still_nan = was_created & grid["M1_cons"].isna()
print(f"\ncreated rows (not observed): {int(was_created.sum()):,}")
print(f"  interpolated: {int(interp.sum()):,} "
      f"(gaps of <= {cfg.cleaning['max_gap_hours']} h)")
print(f"  still NaN: {int(still_nan.sum()):,}")
print(f"gap_flag rows: {int(grid['gap_flag'].sum()):,} "
      f"({grid['gap_flag'].mean() * 100:.1f}% of the grid)")

In [ ]:
# Split the flagged hours into "before the device existed", "after it went silent" and
# "true internal gap" — only the third kind is missing data.
first_last = derived.groupby("device_id")["ts_hour"].agg(["min", "max"])
g = grid.merge(first_last, left_on="device_id", right_index=True, how="left")
lead = g["gap_flag"] & (g["ts_hour"] < g["min"])
trail = g["gap_flag"] & (g["ts_hour"] > g["max"])
inner = g["gap_flag"] & ~lead & ~trail

gap_kinds = pd.DataFrame({
    "kind": ["leading (before first reading)", "trailing (after last reading)",
             "internal (true gap)"],
    "rows": [int(lead.sum()), int(trail.sum()), int(inner.sum())],
})
gap_kinds["share_of_grid_%"] = (gap_kinds["rows"] / len(grid) * 100).round(1)
gap_kinds["share_of_gap_flag_%"] = (gap_kinds["rows"] / int(grid["gap_flag"].sum()) * 100).round(1)
print(gap_kinds)

# Count runs of consecutive flagged hours inside each device's own first->last window.
runs = []
for device_id, gd in g.groupby("device_id"):
    gd = gd.sort_values("ts_hour")
    inside = gd[(gd["ts_hour"] >= gd["min"]) & (gd["ts_hour"] <= gd["max"])]
    run_length = 0
    for is_gap in inside["gap_flag"]:
        if is_gap:
            run_length += 1
        elif run_length > 0:
            runs.append({"device_id": device_id, "length_h": run_length})
            run_length = 0
    if run_length > 0:
        runs.append({"device_id": device_id, "length_h": run_length})
gap_runs = pd.DataFrame(runs)

print("\ninternal gap runs:", len(gap_runs), "across",
      gap_runs["device_id"].nunique(), "devices")
print(gap_runs["length_h"].describe(percentiles=[0.5, 0.9, 0.99]).round(1).to_frame().T)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

# Left: how long the internal gaps that survive interpolation are. One-hour gaps cannot
# appear here — interpolation already filled them.
ax = axes[0]
bins = [2, 3, 5, 9, 25, 49, 169, 721, gap_runs["length_h"].max() + 1]
labels = ["2", "3-4", "5-8", "9-24", "25-48", "2-7 d", "1-30 d", ">30 d"]
binned = pd.cut(gap_runs["length_h"], bins=bins, right=False, labels=labels)
vc = binned.value_counts().reindex(labels).fillna(0)
ax.bar(range(len(vc)), vc.values, color="tab:blue")
ax.set_xticks(range(len(vc)))
ax.set_xticklabels(labels)
ax.set_title("Internal gaps that survive interpolation")
ax.set_xlabel("gap length (consecutive hours)")
ax.set_ylabel("number of gap runs")

# Right: per device, how many flagged hours are leading and how many internal.
ax = axes[1]
gap_kind_per_row = g[["device_id"]].copy()
gap_kind_per_row["lead"] = lead
gap_kind_per_row["inner"] = inner
per_dev = gap_kind_per_row.groupby("device_id")[["lead", "inner"]].sum()
per_dev = per_dev.sort_values("lead", ascending=False)
y = np.arange(len(per_dev))
ax.barh(y, per_dev["lead"].values, color="tab:orange",
        label="leading - device did not exist yet")
ax.barh(y, per_dev["inner"].values, left=per_dev["lead"].values, color="tab:blue",
        label="internal - genuinely missing")
device_labels = []
for device_id in per_dev.index:
    device_labels.append(device_id[-5:])
ax.set_yticks(y)
ax.set_yticklabels(device_labels, fontsize=8)
ax.invert_yaxis()
ax.legend()
ax.set_title("Where every gap_flag hour comes from")
ax.set_xlabel("hours flagged as gap")
ax.set_ylabel("device (last 5 chars of id)")
plt.tight_layout()
plt.show()

#### What a one-hour fill costs

The honest way to price the interpolation is to pay it on hours we actually have. We take
hours whose two neighbours are present, hide the middle one, fill it the way the grid does
(the linear mid-point of its neighbours) and compare with the truth. The comparison is
against a "same as the previous hour" carry-forward, which is the cheapest alternative fill,
and against the plain hour-to-hour variability of the series, which is the error bar any
fill has to live inside.

In [ ]:
# Hide a real hour, fill it, compare with the truth. 400 hours per device, seeded.
rng = np.random.default_rng(cfg.random_seed)
fill_rows = []
for device_id, gd in derived.groupby("device_id"):
    gd = gd.sort_values("ts_hour").reset_index(drop=True)
    ts = gd["ts_hour"]
    # An hour is usable only when both neighbours are exactly one hour away.
    prev_ok = (ts - ts.shift(1)) == pd.Timedelta(hours=1)
    next_ok = (ts.shift(-1) - ts) == pd.Timedelta(hours=1)
    usable = np.flatnonzero((prev_ok & next_ok).to_numpy())
    if len(usable) == 0:
        continue
    sample = rng.choice(usable, size=min(400, len(usable)), replace=False)
    for col in ("grid_import", "grid_export"):
        values = gd[col].to_numpy()
        for i in sample:
            fill_rows.append({
                "device_id": device_id,
                "target": col,
                "truth": values[i],
                "linear_fill": 0.5 * (values[i - 1] + values[i + 1]),
                "carry_forward": values[i - 1],
            })
fills = pd.DataFrame(fill_rows)

rows = []
for target, td in fills.groupby("target"):
    rows.append({
        "target": target,
        "hours_tested": len(td),
        "mean_value_kWh_h": round(float(td["truth"].mean()), 3),
        "MAE_linear_fill": round(float((td["linear_fill"] - td["truth"]).abs().mean()), 4),
        "MAE_carry_forward": round(float((td["carry_forward"] - td["truth"]).abs().mean()), 4),
        "mean_abs_hour_to_hour_change": round(
            float((td["truth"] - td["carry_forward"]).abs().mean()), 4),
    })
cost = pd.DataFrame(rows)
cost["linear_better_by_%"] = (
    100 * (1 - cost["MAE_linear_fill"] / cost["MAE_carry_forward"])).round(1)
print("cost of filling one missing hour, measured on hours we actually have:")
print(cost)
print(f"\nThe fill is applied to {int(interp.sum())} hours out of {len(grid):,} on the grid,")
print("so this error enters the data on a fraction of a per cent of the rows.")

In [ ]:
# One concrete filled hour: the DST hole of 2025-10-26 00:00 UTC on the device with the
# most export among those online that night.
hole_hour = pd.Timestamp("2025-10-26 00:00", tz="UTC")
online_that_night = sorted(night["device_id"].unique())
night_export = derived[derived["device_id"].isin(online_that_night)]
example_dev = night_export.groupby("device_id")["grid_import"].mean().idxmax()

window_start = hole_hour - pd.Timedelta(hours=10)
window_end = hole_hour + pd.Timedelta(hours=10)
is_device = grid["device_id"] == example_dev
in_window = (grid["ts_hour"] >= window_start) & (grid["ts_hour"] <= window_end)
example = grid[is_device & in_window].sort_values("ts_hour")
filled = example[~example["was_observed"]]

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(example["ts_hour"], example["grid_import"], color="tab:blue", marker="o",
        markersize=4, label="grid import (hourly)")
ax.scatter(filled["ts_hour"], filled["grid_import"], s=90, color="tab:orange", zorder=5,
           label="hour created and linearly filled")
ax.xaxis.set_major_formatter(mpl.dates.DateFormatter("%d %b %H:%M"))
ax.tick_params(axis="x", rotation=25)
ax.set_title(f"The DST fall-back hole, filled - device ...{example_dev[-5:]}")
ax.set_xlabel("time (UTC)")
ax.set_ylabel("grid import (kWh per hour)")
ax.legend()
plt.tight_layout()
plt.show()

print("device:", example_dev)
print("hours created in this window:", list(filled["ts_hour"]))
print("filled values:", filled[["grid_import", "grid_export"]].round(4).to_dict("records"))

#### We keep each device on its own first→last hour

The leading and trailing rows carry no information at all — every value in them is NaN —
yet they dominate the frame and distort every "share of hours flagged" statistic. We
therefore trim each device to its own first→last observed hour and work on that from here
on. The internal gaps are untouched: the only rows that go are rows that never existed.

In [ ]:
def trim_to_device_span(df, cols=("M1_cons", "M1_prod")):
    """Drop leading/trailing all-NaN rows per device, keeping internal gaps."""
    out = []
    for _, gd in df.groupby("device_id", sort=True):
        gd = gd.sort_values("ts_hour")
        # A row counts as "seen" when at least one raw meter column has a value.
        seen = gd[list(cols)].notna().any(axis=1)
        if not seen.any():
            continue
        first_label = seen[seen].index[0]
        last_label = seen[seen].index[-1]
        out.append(gd.loc[first_label:last_label])
    return pd.concat(out).reset_index(drop=True)


grid_trimmed = trim_to_device_span(grid)
print(f"rows on the global grid: {len(grid):,}")
print(f"rows after the per-device trim: {len(grid_trimmed):,} "
      f"(-{(1 - len(grid_trimmed) / len(grid)) * 100:.1f}%)")
print(f"gap_flag share on the global grid: {grid['gap_flag'].mean() * 100:.1f}%")
print(f"gap_flag share after trimming: {grid_trimmed['gap_flag'].mean() * 100:.1f}%")
print("\nSame data, same internal gaps: what goes is rows in which the device did not exist.")

### 2.4 Weather preparation and the merge

`prepare_weather` normalises the weather index to tz-aware UTC (a naive input would be read
as `config.local_tz`, Open-Meteo's `timezone=auto` convention), drops the ambiguous DST
fall-back hour, deduplicates, and adds one derived feature: **`ghi_ramp`**, the
hour-over-hour change in tilted irradiance. A ramp feature matters because a PV meter's
export at a given irradiance depends on whether the sky is opening or closing.

The extract is already tz-aware UTC, so the localisation branch is a no-op here — worth
showing rather than assuming. The merge is a left join on the hour, so coverage is measured
on the trimmed grid afterwards.

In [ ]:
w = clean.prepare_weather(weather, cfg)
print(f"weather rows in: {len(weather):,}, out: {len(w):,}")
print(f"index: {w.index.name}, tz={w.index.tz}, "
      f"monotonic={w.index.is_monotonic_increasing}")
print("span:", w.index.min(), "->", w.index.max())
print(f"added column ghi_ramp: range {w['ghi_ramp'].min():.0f} .. "
      f"{w['ghi_ramp'].max():.0f} W/m^2 per hour")

# Hourly completeness of the weather frame itself.
expected = pd.date_range(w.index.min(), w.index.max(), freq="h")
print(f"hourly completeness: {len(w)}/{len(expected)} = {100 * len(w) / len(expected):.2f}%")

nan_share = (w.isna().mean() * 100).round(2).rename("nan_%")
print("\nNaN share per weather column (%):")
if (nan_share > 0).any():
    print(nan_share[nan_share > 0].to_frame().T)
else:
    print("  none - every column is complete over the whole span")

In [ ]:
# Left join one weather row onto every (device, hour) row of the trimmed grid.
merged = grid_trimmed.merge(w.reset_index(), left_on="ts_hour", right_on="datetime",
                            how="left")
merged = merged.drop(columns=["datetime"])
print(f"rows before merge: {len(grid_trimmed):,}, after: {len(merged):,} (left join, 1:1)")

weather_cols = list(cfg.features["weather_all"])
coverage = merged[weather_cols].notna().mean().mul(100).round(2)
print("\nweather coverage on the trimmed grid (% of rows with a value):")
print(coverage.to_frame("coverage_%").T)
print("\nminimum coverage across all", len(weather_cols), "columns:",
      f"{coverage.min():.2f}%")
assert float(coverage.min()) == 100.0, "a weather column does not cover the meter grid"
print("Every weather column covers every hour of every device: no imputation is needed and")
print("no model row trains without weather.")

### 2.5 Calendar features — the local clock and the cyclic hour

Calendar features are computed on **local** time (`config.local_tz`), not UTC: the load
shape follows the human day and the sun, both of which move with DST. `hour_sin` /
`hour_cos` encode the hour on a circle so that 23:00 and 00:00 are neighbours, which a raw
integer hour cannot express to a tree that splits on thresholds.

`theoretical_prod = global_tilted_irradiance * effective_solar_pv` is a physical prior: the
shape a perfect PV plant would produce, before anything about this particular device.

Two checks: the cyclic encoding really traces a circle over a week, and `hour_local` really
is local — if it were UTC, the export peak would sit an hour or two early.

In [ ]:
cal = clean.add_calendar_features(merged, cfg)

added = []
for col in cal.columns:
    if col not in merged.columns:
        added.append(col)
print("columns added:", added)
print("hour_local dtype", cal["hour_local"].dtype, "range",
      cal["hour_local"].min(), "..", cal["hour_local"].max())
print("is_daylight coerced to int:", sorted(cal["is_daylight"].dropna().unique()))

# One week of one device, used here and in the next figure.
one_dev = cal["device_id"].iloc[0]
wk_start = pd.Timestamp("2026-06-15", tz="UTC")
wk_end = wk_start + pd.Timedelta(days=7)
in_week = (cal["ts_hour"] >= wk_start) & (cal["ts_hour"] < wk_end)
week = cal[(cal["device_id"] == one_dev) & in_week]
print(f"\nexample week: {wk_start:%Y-%m-%d} -> {wk_end:%Y-%m-%d} ({len(week)} hourly rows)")
week[["ts_hour", "hour_local", "day_of_week", "is_weekend", "hour_sin", "hour_cos",
      "theoretical_prod"]].head(6).round(3)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))

# Left: the two cyclic components over one week.
ax = axes[0]
ax.plot(week["ts_hour"], week["hour_sin"], color="tab:blue", label="hour_sin")
ax.plot(week["ts_hour"], week["hour_cos"], color="tab:orange", label="hour_cos")
ax.legend()
ax.tick_params(axis="x", rotation=25)
ax.xaxis.set_major_formatter(mpl.dates.DateFormatter("%d %b"))
ax.set_title("Cyclic hour encoding - one week")
ax.set_xlabel("date (UTC)")
ax.set_ylabel("value (dimensionless)")

# Middle: the same two components plotted against each other.
ax = axes[1]
ax.plot(week["hour_sin"], week["hour_cos"], color="tab:blue", marker="o")
ax.set_aspect("equal")
ax.set_title("...is a circle, so 23:00 neighbours 00:00")
ax.set_xlabel("hour_sin")
ax.set_ylabel("hour_cos")

# Right: the cohort mean day, to confirm hour_local really is the local clock.
ax = axes[2]
prof = cal.groupby("hour_local")[["grid_export", "grid_import"]].mean()
ax.plot(prof.index, prof["grid_export"], color="tab:orange", label="grid export")
ax.plot(prof.index, prof["grid_import"], color="tab:blue", label="grid import")
peak = int(prof["grid_export"].idxmax())
ax.axvspan(12, 14, color="green", alpha=0.15)
ax.set_xticks(range(0, 24, 3))
ax.legend()
ax.set_title("Cohort mean profile on the local clock")
ax.set_xlabel("hour of day (Europe/Rome)")
ax.set_ylabel("energy (kWh per hour)")
plt.tight_layout()
plt.show()
print(f"grid_export peaks at local hour {peak} (12-14 expected for a south-facing site),")
print("which confirms hour_local is the local clock and not UTC.")

### 2.6 Outlier flags — a centred rolling z-score

For each target we compute a **centred** 168 h rolling mean and standard deviation per
device (min 24 periods) and flag `|z| > 3`. The flags are informational: nothing downstream
drops them.

Two caveats worth stating plainly. The window is **centred**, so the flag for hour *t* uses
hours after *t* — fine for a retrospective quality report, a leak if it ever became a
feature. And both targets are **zero-inflated**: at night a PV device exports exactly zero
for hours on end, so the local rolling standard deviation collapses towards the `1e-6`
guard and the z-score explodes on any non-zero value. The flags therefore cluster at the
shoulders of the solar day rather than on implausible readings.

In [ ]:
flagged = clean.add_outlier_flags(cal, cfg)
win = cfg.cleaning["outlier_window_hours"]
thr = cfg.cleaning["outlier_zscore_threshold"]
print(f"rolling window {win} h (centred), min_periods "
      f"{cfg.cleaning['outlier_min_periods']}, |z| > {thr}")

rows = []
for col in ("grid_export", "grid_import"):
    flags = flagged[f"{col}_outlier"]
    is_observed = flagged[col].notna()
    is_nonzero = flagged[col].fillna(0) > 0
    flag_count = int(flags.sum())
    rows.append({
        "target": col,
        "flagged": flag_count,
        "share_of_observed_%": round(flags.sum() / is_observed.sum() * 100, 2),
        "flagged_that_are_zero_%": round((flags & ~is_nonzero).sum() / max(flag_count, 1) * 100, 1),
        "devices_with_flags": int(flagged.loc[flags, "device_id"].nunique()),
        "median_flagged_value": round(float(flagged.loc[flags, col].median()), 3),
        "median_value_overall": round(float(flagged.loc[is_observed, col].median()), 3),
    })
pd.DataFrame(rows)

In [ ]:
# The device with the most export flags, one month of it.
cand = flagged.groupby("device_id")["grid_export_outlier"].sum().sort_values(ascending=False)
dev_out = cand.index[0]
sub = flagged[flagged["device_id"] == dev_out].sort_values("ts_hour")

# Centre the window on the median flagged hour, 15 days on each side.
mid = sub.loc[sub["grid_export_outlier"], "ts_hour"].median()
sub = sub[(sub["ts_hour"] >= mid - pd.Timedelta(days=15))
          & (sub["ts_hour"] <= mid + pd.Timedelta(days=15))]
out = sub[sub["grid_export_outlier"]]

fig, axes = plt.subplots(2, 1, figsize=(11, 5.4))

# Top: the series with the flagged points on top.
ax = axes[0]
ax.plot(sub["ts_hour"], sub["grid_export"], color="tab:blue", label="grid export")
ax.scatter(out["ts_hour"], out["grid_export"], color="tab:orange",
           label=f"flagged |z| > {thr}")
ax.legend()
ax.xaxis.set_major_formatter(mpl.dates.DateFormatter("%d %b"))
ax.set_title(f"Rolling z-score flags - device ...{dev_out[-5:]}, 30 days")
ax.set_xlabel("date (UTC)")
ax.set_ylabel("grid export (kWh per hour)")

# Bottom: flag rate by hour of day, against how often the series is non-zero at all.
ax = axes[1]
hourly_flags = pd.DataFrame({
    "hour_local": flagged["hour_local"],
    "grid_export_outlier": flagged["grid_export_outlier"],
    "nz": flagged["grid_export"].fillna(0) > 0,
})
by_hour = hourly_flags.groupby("hour_local").agg(flag_rate=("grid_export_outlier", "mean"),
                                                 nonzero_rate=("nz", "mean"))
x = np.arange(24)
ax.bar(x - 0.2, by_hour["flag_rate"].values * 100, width=0.4, color="tab:orange",
       label="hours flagged as outliers (%)")
ax.bar(x + 0.2, by_hour["nonzero_rate"].values * 100, width=0.4, color="tab:blue",
       label="hours with export > 0 (%)")
ax.set_xticks(range(0, 24, 2))
ax.legend()
ax.set_title("When the flags fire, against when the series is even non-zero")
ax.set_xlabel("hour of day (Europe/Rome)")
ax.set_ylabel("share of hours (%)")
plt.tight_layout()
plt.show()
print("device with most export flags:", dev_out, f"({int(cand.iloc[0])} flags)")
print("The flags concentrate at the shoulders of the solar day, where the rolling std")
print("collapses on a mostly-zero series.")

### 2.7 The same chain in one call

`build_processed_hourly` is the one-call version, and it is what the command-line pipeline
runs. It must reproduce the step-by-step result exactly; the two must agree or the
walkthrough above describes something the pipeline does not do. The comparison is on the
rows they share, since the step-by-step frame was trimmed per device in 2.3.

In [ ]:
processed = clean.build_processed_hourly(df_meters, cfg, df_weather=weather)
print("build_processed_hourly ->", processed.shape)

# Put the step-by-step frame in the same shape as the one-call one.
step_by_step = flagged.copy()
step_by_step["gap_flag"] = flagged["gap_flag"].fillna(False).astype(bool)
step_by_step = step_by_step.sort_values(["device_id", "ts_hour"]).reset_index(drop=True)
step_by_step = step_by_step.drop(columns=["was_observed"], errors="ignore")

print("columns identical:", list(processed.columns) == list(step_by_step.columns))
print(f"one-call rows: {len(processed):,} (global grid)")
print(f"step-by-step rows: {len(step_by_step):,} (trimmed per device in 2.3)")

key = ["device_id", "ts_hour"]
a = processed.merge(step_by_step[key], on=key, how="inner")
a = a.sort_values(key).reset_index(drop=True)
b = step_by_step.sort_values(key).reset_index(drop=True)

num_cols = []
bool_cols = []
for col in processed.columns:
    if pd.api.types.is_numeric_dtype(processed[col]):
        num_cols.append(col)
    if processed[col].dtype == bool:
        bool_cols.append(col)

pd.testing.assert_frame_equal(a[num_cols], b[num_cols], rtol=1e-10, atol=1e-12)
for col in bool_cols:
    assert a[col].equals(b[col]), "boolean columns differ"
print(f"\nassert_frame_equal PASSED on {len(num_cols)} numeric + {len(bool_cols)} boolean "
      f"columns over {len(a):,} shared rows")

PT = trim_to_device_span(processed)
print("\nprocessed frame used from here on (trimmed):", PT.shape)
print("span:", PT["ts_hour"].min(), "->", PT["ts_hour"].max())

## 3. Per-device data-quality table

One row per meter, joining everything measured above: how much data there is, how much of
it was filled in, how much is flagged, and whether the meter exports at all.

The **PV flag** is computed from the data: a device whose mean `grid_export` is above
`sufficiency.export_min_mean_kwh` (0.01 kWh/h) has photovoltaic production worth
forecasting. It matters twice. The export model only makes sense for a device that exports,
and the import model's physics differs with PV: for a device with panels, import is "how
much the sun did *not* cover", which makes irradiance a *negative* regressor; for a device
without, import is mostly a heating problem.

The table is written to `notebooks/data/device_quality.csv`.

In [ ]:
# Per-device counts measured on the earlier, untrimmed frames.
interp_by_device = pd.DataFrame({"device_id": grid["device_id"], "interp": interp})
interp_per_dev = interp_by_device.groupby("device_id")["interp"].sum()
partial_per_dev = hourly.groupby("device_id")["partial_hour"].sum()
gap_kind_by_device = pd.DataFrame({"device_id": g["device_id"], "lead": lead, "inner": inner})
inner_per_dev = gap_kind_by_device.groupby("device_id")["inner"].sum()
lead_per_dev = gap_kind_by_device.groupby("device_id")["lead"].sum()

q = PT.groupby("device_id").agg(
    rows=("ts_hour", "size"),
    first_ts=("ts_hour", "min"),
    last_ts=("ts_hour", "max"),
    mean_export_kwh_h=("grid_export", "mean"),
    mean_import_kwh_h=("grid_import", "mean"),
    gap_rows=("gap_flag", "sum"),
    export_outliers=("grid_export_outlier", "sum"),
    import_outliers=("grid_import_outlier", "sum"),
)
q["span_days"] = ((q["last_ts"] - q["first_ts"]).dt.total_seconds() / 86400).round(1)
q["coverage"] = (1 - q["gap_rows"] / q["rows"]).round(4)
q["interpolated_rows"] = interp_per_dev.reindex(q.index).fillna(0).astype(int)
q["partial_hours"] = partial_per_dev.reindex(q.index).fillna(0).astype(int)
q["internal_gap_rows"] = inner_per_dev.reindex(q.index).fillna(0).astype(int)
q["grid_padding_rows_dropped"] = lead_per_dev.reindex(q.index).fillna(0).astype(int)

# The PV flag, from the data and the configured threshold.
export_floor = float(cfg.sufficiency["export_min_mean_kwh"])
q["has_pv"] = q["mean_export_kwh_h"] > export_floor
q = q.round(4).sort_values("span_days", ascending=False)

HAS_PV = q["has_pv"].to_dict()
print(f"PV flag: mean grid_export > {export_floor} kWh/h")
print(f"  with PV: {int(q['has_pv'].sum())} devices")
print("  consumption only:", list(q.index[~q["has_pv"]].str[-5:]))

out_csv = DATA_DIR / "device_quality.csv"
q.to_csv(out_csv)
print("\nwrote", out_csv, f"({len(q)} rows x {q.shape[1]} cols)")
print(f"totals: {int(q['rows'].sum()):,} hourly rows, "
      f"{int(q['interpolated_rows'].sum()):,} interpolated, "
      f"{int(q['partial_hours'].sum()):,} partial hours, "
      f"{int(q['export_outliers'].sum()) + int(q['import_outliers'].sum()):,} outlier flags")

q_display = q.copy()
q_display["device"] = q.index.str[-5:]
q_display = q_display.set_index("device")
q_display[["rows", "span_days", "coverage", "mean_import_kwh_h", "mean_export_kwh_h",
           "has_pv", "gap_rows", "internal_gap_rows", "interpolated_rows", "partial_hours",
           "export_outliers", "import_outliers"]]

## 4. The features

### 4.1 The forecasting setup

We train **one model per (device, target)**. A model is asked, at a forecast origin `o`,
for the next 48 hours: the target hours are `t = o + h` for `h = 1..48`.

The trick that keeps this to one model instead of 48 is that **each training row is one
`(t, h)` pair** and `horizon` is itself a feature. A target hour that appears once in the
series appears once per horizon in the training matrix, each time with the history that
would have been visible `h` hours earlier. The model therefore learns how the relationship
between the history and the target degrades as the forecast reaches further out, and one
booster serves every horizon and every origin.

Training uses a sparse set of horizons — seventeen of the forty-eight — because
neighbouring horizons carry nearly the same information and the matrix grows linearly in
their number. Forecasting still covers all 48.

### 4.2 The observability rule

This is the part where a forecast quietly becomes a lie if it is wrong.

A feature derived from the target may only read values **at or before the origin**. A lag
of `L` hours from the target hour points at `t - L = o + h - L`, which the origin has seen
only when

> **`L >= h`**

Otherwise the value lies *after* the origin and does not exist at forecast time. Training on
it would flatter the model exactly where it is weakest. Two consequences run through the
whole catalogue below: `same_hour_1d` falls back from a 24 h to a 48 h offset once
`h > 24`, and the averages over day lags use only the day lags `d` with `24d >= h`.

Weather and calendar features are read at the **target hour** `t`, not at the origin,
because both are known from the forecast when the origin is set. For weather we use the
archive, which amounts to **assuming a perfect weather forecast**. That is an optimistic
assumption and it is stated once here: the numbers in section 5 are an upper bound on what
the weather features can contribute in production.

### 4.3 The feature catalogue

Fifty-one features in four families. All of them go into the pool from the start; section 5
decides which stay.

#### Calendar (9) — read at the target hour, on the local clock

| feature | how it is computed | why we expect it to help |
|---|---|---|
| `hour_sin` | `sin(2π · hour_local / 24)` | the hour of the day as a circle, so 23:00 and 00:00 are neighbours |
| `hour_cos` | `cos(2π · hour_local / 24)` | the second coordinate of the same circle |
| `day_of_week` | local weekday, 0 = Monday | weekday and weekend load shapes differ |
| `month` | local month, 1–12 | a coarse season index |
| `is_weekend` | `day_of_week ∈ {5, 6}` | the weekday/weekend split, made explicit for a tree |
| `doy_sin` | `sin(2π · day_of_year / 365.25)` | the season as a circle: 31 Dec and 1 Jan are neighbours, which `month` cannot express |
| `doy_cos` | `cos(2π · day_of_year / 365.25)` | the second coordinate of the season circle |
| `is_holiday` | local date in the Italian national holiday list (2025–2026, Easter Monday 21 Apr 2025 and 6 Apr 2026) | a holiday looks like a Sunday to the load and like a Tuesday to `day_of_week` |
| `is_bridge` | a working day wedged between a holiday and a weekend | in Italy a bridge day empties buildings as reliably as the holiday itself |

#### Weather at the target hour (24) — assumed known from the forecast

The first thirteen come straight out of the Open-Meteo download of notebook 01
(`build_weather_features`) and the weather preparation of 2.4; the last eleven we derive
here.

| feature | how it is computed | why we expect it to help |
|---|---|---|
| `global_tilted_irradiance` | Open-Meteo, panel tilt 30°, azimuth 0 (south), W/m² | the irradiance a south-facing panel actually receives — the first-order driver of export |
| `shortwave_radiation` | Open-Meteo global horizontal irradiance, W/m² | the same energy on a horizontal plane; a second view of the same signal |
| `cloud_cover` | Open-Meteo, % | what stands between the panel and the sun |
| `temperature_2m` | Open-Meteo, °C | heating and cooling demand drive import |
| `solar_elevation` | NOAA solar geometry for the site, clipped at 0 so night is 0, degrees | pure geometry: where the sun is, independent of weather |
| `effective_solar_pv` | `cos(solar zenith)` clipped to `[0, 1]` | the geometric, cloud-free PV availability of the hour |
| `clearsky_index` | `shortwave_radiation / clear-sky GHI`, clipped to `[0, 1.2]`, with clear-sky GHI from Haurwitz `1098 · cz · exp(−0.059 / cz)` | turns irradiance into "how much of what was available", which is comparable across seasons |
| `heating_degree` | `max(18 − T, 0)` | the part of the temperature that costs heating energy |
| `cooling_degree` | `max(T − 24, 0)` | the part that costs cooling energy |
| `pv_temp_factor` | `1 − 0.004 · max(T − 25, 0)` | PV derating with cell temperature: a hot panel produces less |
| `is_daylight` | Open-Meteo `is_day` flag, 0/1 | a hard day/night switch a tree can split on in one go |
| `cloud_cover_diff` | signed hour-over-hour change of `cloud_cover` | a sky that is opening behaves differently from one that is closing |
| `ghi_ramp` | signed hour-over-hour change of `global_tilted_irradiance` | the same idea on the quantity that actually reaches the panel |
| `theoretical_prod` | `global_tilted_irradiance × effective_solar_pv` | the shape a perfect plant would produce, before anything about this device |
| `clearsky_ghi` | Haurwitz clear-sky GHI from the site's solar geometry, W/m² | the physical ceiling of the hour, the denominator of `clearsky_index` made explicit |
| `gti_roll_3h` | 3 h rolling mean of `global_tilted_irradiance` | a smoother regressor than the spot value, robust to a single mis-timed cloud |
| `cloud_std_6h` | 6 h rolling standard deviation of `cloud_cover` | volatility: a stable overcast and a broken sky can share the same mean cover |
| `temp_lag_24h` | `temperature_2m` 24 h earlier | thermal inertia: a building answers to yesterday's temperature too |
| `temp_roll_24h` | 24 h rolling mean of `temperature_2m` | the same inertia, smoothed |
| `gti_day_sum` | total `global_tilted_irradiance` over the target's **local** day | gives every hour a view of the whole day's weather |
| `temp_range_day` | daily max − min of `temperature_2m` over the local day | a proxy for clear versus overcast synoptics |
| `day_length_h` | sunset − sunrise for the local date, from a 15-minute solar-elevation grid | how long the solar day is, which `hour_local` cannot express across seasons |
| `hours_since_sunrise` | `t − sunrise` of the local date, hours | where the hour sits inside its own solar day |
| `hours_to_sunset` | `sunset − t` of the local date, hours | the same, measured from the other end |

#### Target history (17) — every one satisfies `L >= h`

Named `<target>_<name>`, so `grid_export_same_hour_7d` and so on.

| feature | how it is computed | why we expect it to help |
|---|---|---|
| `same_hour_1d` | value at `t − 24 h`, falling back to `t − 48 h` when `h > 24` | yesterday at this hour is the strongest single predictor of today at this hour |
| `same_hour_2d` | value at `t − 48 h` | the same echo one day further back, still usable at every horizon |
| `same_hour_3d` | value at `t − 72 h` | |
| `same_hour_7d` | value at `t − 168 h` | the weekly echo: same hour, same weekday |
| `same_hour_14d` | value at `t − 336 h` | two weeks back, less noisy than one |
| `same_hour_21d` | value at `t − 504 h` | |
| `same_hour_28d` | value at `t − 672 h` | a monthly echo, and a slow seasonal level |
| `mean_same_hour_7d` | mean over the day lags `d` with `24d >= h` of the value at `t − 24d` | the typical level of this hour, averaged over the last week |
| `median_same_hour_7d` | median over the same observable day lags | the robust twin: one bad day moves a 7-point mean by a seventh and barely moves the median |
| `diff_1d` | `same_hour_1d − same_hour_2d` | the day-over-day trend of this hour |
| `diff_7d` | `same_hour_7d − same_hour_14d` | the week-over-week trend |
| `roll_24h_mean` | mean of the 24 h window **ending at the origin** (min 12 values) | the device's current level, the last thing the operator actually saw averaged over a day |
| `roll_24h_std` | standard deviation of the same window | how erratic the device has been lately |
| `roll_24h_max` | maximum of the same window | recent peak capability |
| `value_at_origin` | value at `t − h`, i.e. the last observed hour | the single freshest observation |
| `prev_day_total` | sum of the 24 h window ending at the origin (min 12 values) | the device's recent daily energy |
| `zero_share_7d` | share of exactly-zero hours in the 168 h before the origin (min 24 values) | an activity detector: a device silent for a week is probably still silent |

#### Horizon (1)

| feature | how it is computed | why we expect it to help |
|---|---|---|
| `horizon` | `h`, the number of hours between origin and target hour | lets one booster serve every horizon: it learns how far the history can be trusted |

#### Two things we deliberately leave out

Device-level constants such as a PV flag or a per-device export percentile are **constant
within a device**, and every model here is trained on a single device. A constant column
cannot split, so it cannot help; it would only mean something in a model pooled across
devices.

#### A known duplication

For `h > 24` the fallback makes `same_hour_1d` read `t − 48 h`, which is exactly what
`same_hour_2d` reads. The two columns are then identical and `diff_1d` is identically zero
across the whole 25–48 h range: a feature that cannot split, and a duplicate that dilutes
the column subsampling. We leave both in the pool and let section 5 confirm it.

### 4.4 The code

One function builds the matrix. The weather-derived columns do not depend on the device, so
they are computed once for every hour of the weather frame and mapped in; the sunrise and
sunset of each local date come from a 15-minute grid of solar elevations.

In [ ]:
from celine.forecasting.core.weather import _haurwitz_clearsky_ghi, solar_position
from celine.forecasting.models.lightgbm.features import observable_day_lags

# The prepared weather frame, on a tz-aware UTC index.
WX = clean.prepare_weather(weather, cfg).sort_index()

# WD: the weather-derived columns, computed once for every hour.
WD = pd.DataFrame(index=WX.index)
_, cos_zenith = solar_position(WX.index, SITE_LAT, SITE_LON)
WD["clearsky_ghi"] = _haurwitz_clearsky_ghi(cos_zenith)
WD["gti_roll_3h"] = WX["global_tilted_irradiance"].rolling(3, min_periods=1).mean()
WD["cloud_std_6h"] = WX["cloud_cover"].rolling(6, min_periods=2).std()
WD["temp_lag_24h"] = WX["temperature_2m"].shift(24)
WD["temp_roll_24h"] = WX["temperature_2m"].rolling(24, min_periods=6).mean()

# Whole-local-day aggregates, attached back to every hour of that day.
local_date = pd.Series(WX.index.tz_convert(LOCAL_TZ).date, index=WX.index)
gti_by_day = WX.groupby(local_date.values)["global_tilted_irradiance"].sum()
WD["gti_day_sum"] = local_date.map(gti_by_day).values
temp_max_by_day = WX.groupby(local_date.values)["temperature_2m"].max()
temp_min_by_day = WX.groupby(local_date.values)["temperature_2m"].min()
WD["temp_range_day"] = local_date.map(temp_max_by_day - temp_min_by_day).values

# SD: sunrise, sunset and day length per local date, from a 15-minute elevation grid.
fine = pd.date_range(WX.index.min().floor("D") - pd.Timedelta(days=1),
                     WX.index.max().ceil("D") + pd.Timedelta(days=1),
                     freq="15min", tz="UTC")
fine_elevation, _ = solar_position(fine, SITE_LAT, SITE_LON)
minutes = pd.DataFrame({"d": fine.tz_convert(LOCAL_TZ).date, "ts": fine}, index=fine)
lit = minutes[fine_elevation > 0]
SD = lit.groupby("d").agg(sunrise=("ts", "min"), sunset=("ts", "max"))
SD["day_length_h"] = (SD["sunset"] - SD["sunrise"]).dt.total_seconds() / 3600

ITALIAN_HOLIDAYS = [
    "2025-01-01", "2025-01-06", "2025-04-21", "2025-04-25", "2025-05-01", "2025-06-02",
    "2025-08-15", "2025-11-01", "2025-12-08", "2025-12-25", "2025-12-26",
    "2026-01-01", "2026-01-06", "2026-04-06", "2026-04-25", "2026-05-01", "2026-06-02",
    "2026-08-15", "2026-11-01", "2026-12-08", "2026-12-25", "2026-12-26",
]
HOLIDAYS = set()
for day in ITALIAN_HOLIDAYS:
    HOLIDAYS.add(pd.Timestamp(day).date())

# A bridge day is a working day next to a holiday whose other neighbour is also off.
BRIDGES = set()
for day in sorted(HOLIDAYS):
    ts = pd.Timestamp(day)
    for step in (-1, 1):
        candidate = ts + pd.Timedelta(days=step)
        if candidate.date() in HOLIDAYS or candidate.weekday() >= 5:
            continue
        next_day = candidate + pd.Timedelta(days=step)
        if next_day.date() in HOLIDAYS or next_day.weekday() >= 5:
            BRIDGES.add(candidate.date())

# EXTRA: everything that depends only on the target hour, in one frame keyed by that hour.
HOURS = WX.index
hours_local = HOURS.tz_convert(LOCAL_TZ)
EXTRA = pd.DataFrame(index=HOURS)
day_of_year = hours_local.dayofyear.to_numpy()
EXTRA["doy_sin"] = np.sin(2 * np.pi * day_of_year / 365.25)
EXTRA["doy_cos"] = np.cos(2 * np.pi * day_of_year / 365.25)
hour_dates = pd.Series(hours_local.date, index=HOURS)
EXTRA["is_holiday"] = hour_dates.isin(HOLIDAYS).astype(int).values
EXTRA["is_bridge"] = hour_dates.isin(BRIDGES).astype(int).values
for col in WD.columns:
    EXTRA[col] = WD[col].values
EXTRA["day_length_h"] = hour_dates.map(SD["day_length_h"]).values
sunrise_of_hour = pd.to_datetime(hour_dates.map(SD["sunrise"]), utc=True)
sunset_of_hour = pd.to_datetime(hour_dates.map(SD["sunset"]), utc=True)
hour_series = pd.Series(HOURS, index=HOURS)
EXTRA["hours_since_sunrise"] = (hour_series - sunrise_of_hour).dt.total_seconds().values / 3600
EXTRA["hours_to_sunset"] = (sunset_of_hour - hour_series).dt.total_seconds().values / 3600

bridge_labels = []
for bridge_day in sorted(BRIDGES):
    bridge_labels.append(str(bridge_day))
print("holidays listed:", len(HOLIDAYS), "- bridge days:", len(BRIDGES), bridge_labels)
print(f"day length at the site: {SD['day_length_h'].min():.1f} h (winter) .. "
      f"{SD['day_length_h'].max():.1f} h (summer)")
print("EXTRA:", EXTRA.shape, "columns:", list(EXTRA.columns))

In [ ]:
# The four families, in the order they enter the matrix.
CALENDAR_FEATURES = ["hour_sin", "hour_cos", "day_of_week", "month", "is_weekend",
                     "doy_sin", "doy_cos", "is_holiday", "is_bridge"]
# The thirteen columns that come with the prepared weather frame, plus theoretical_prod.
WEATHER_FROM_FRAME = ["global_tilted_irradiance", "shortwave_radiation", "cloud_cover",
                      "temperature_2m", "solar_elevation", "effective_solar_pv",
                      "clearsky_index", "heating_degree", "cooling_degree",
                      "pv_temp_factor", "is_daylight", "cloud_cover_diff", "ghi_ramp",
                      "theoretical_prod"]
# The eleven we derive in EXTRA (doy_* / is_holiday / is_bridge live there too, but they
# are calendar, not weather, so they are listed above).
WEATHER_DERIVED = ["clearsky_ghi", "gti_roll_3h", "cloud_std_6h", "temp_lag_24h",
                   "temp_roll_24h", "gti_day_sum", "temp_range_day", "day_length_h",
                   "hours_since_sunrise", "hours_to_sunset"]
WEATHER_FEATURES = WEATHER_FROM_FRAME + WEATHER_DERIVED
HISTORY_FEATURES = ["same_hour_1d", "same_hour_2d", "same_hour_3d", "same_hour_7d",
                    "same_hour_14d", "same_hour_21d", "same_hour_28d",
                    "mean_same_hour_7d", "median_same_hour_7d", "diff_1d", "diff_7d",
                    "roll_24h_mean", "roll_24h_std", "roll_24h_max", "value_at_origin",
                    "prev_day_total", "zero_share_7d"]

# The columns EXTRA contributes, split by the family they belong to.
EXTRA_CALENDAR = ["doy_sin", "doy_cos", "is_holiday", "is_bridge"]
BASE_CALENDAR = ["hour_sin", "hour_cos", "day_of_week", "month", "is_weekend"]

# The day lags the same-hour features need: 1..7 for the observable averages, plus the
# three long echoes.
DAY_LAGS = [1, 2, 3, 4, 5, 6, 7, 14, 21, 28]


def feature_names(target):
    """The full ordered feature pool for a target, history columns prefixed."""
    names = list(CALENDAR_FEATURES) + list(WEATHER_FEATURES)
    for name in HISTORY_FEATURES:
        names.append(f"{target}_{name}")
    names.append("horizon")
    return names


FEATURE_FAMILY = {}
for name in CALENDAR_FEATURES:
    FEATURE_FAMILY[name] = "calendar"
for name in WEATHER_FEATURES:
    FEATURE_FAMILY[name] = "weather"
for target in cfg.targets:
    for name in HISTORY_FEATURES:
        FEATURE_FAMILY[f"{target}_{name}"] = "history"
FEATURE_FAMILY["horizon"] = "horizon"

for target in cfg.targets:
    print(f"{target}: {len(feature_names(target))} features "
          f"({len(CALENDAR_FEATURES)} calendar + {len(WEATHER_FEATURES)} weather + "
          f"{len(HISTORY_FEATURES)} history + 1 horizon)")

In [ ]:
def build_feature_matrix(df_device, target, horizons, train_end):
    """Build the (X, y) training matrix of one device for one target.

    Each row is one (target hour t, horizon h) pair. Calendar and weather are read
    at t, because they are known from the forecast; every history feature is read at
    or before the origin t - h, so the observability rule L >= h holds by
    construction.

    Args:
        df_device: Processed hourly frame of a single device.
        target: Target column, ``grid_export`` or ``grid_import``.
        horizons: Horizons to expand the matrix over.
        train_end: Only target hours at or before this timestamp are used.

    Returns:
        ``(X, y)`` with ``ts_hour`` as the first column of ``X``, then every feature
        of the pool; ``y`` is the target at ``ts_hour``. Rows whose target is missing
        are dropped.
    """
    base = df_device[df_device["ts_hour"] <= train_end]
    base = base.sort_values("ts_hour").reset_index(drop=True)
    if base.empty:
        return pd.DataFrame(), pd.Series(dtype=float)

    t = base["ts_hour"]

    # The target-hour block is the same for every horizon, so build it once.
    block = base[["ts_hour"] + BASE_CALENDAR + WEATHER_FROM_FRAME].copy()
    for col in EXTRA.columns:
        block[col] = t.map(EXTRA[col]).values

    # The target as a series indexed by the hour, so any timestamp can be looked up.
    history = pd.Series(base[target].values, index=t)
    history = history[~history.index.duplicated(keep="last")]

    # Rolling views of the history. They are backward-looking, so reading them AT the
    # origin gives the window that ends at the origin.
    roll_mean = history.rolling("24h", min_periods=12).mean()
    roll_std = history.rolling("24h", min_periods=12).std()
    roll_max = history.rolling("24h", min_periods=12).max()
    roll_sum = history.rolling("24h", min_periods=12).sum()
    is_zero = (history == 0).where(history.notna()).astype(float)
    zero_share = is_zero.rolling("168h", min_periods=24).mean()

    prefix = f"{target}_"
    frames = []
    for h in horizons:
        hdf = block.copy()
        hdf["horizon"] = h
        origin = t - pd.Timedelta(hours=h)

        # Every same-hour day lag, looked up once and reused below.
        day_values = {}
        for d in DAY_LAGS:
            day_values[d] = (t - pd.Timedelta(hours=24 * d)).map(history).values

        # A 24 h offset is only observable while h <= 24; beyond that fall back to 48 h.
        if h <= 24:
            offset_1d = 24
        else:
            offset_1d = 48
        hdf[prefix + "same_hour_1d"] = (t - pd.Timedelta(hours=offset_1d)).map(history).values
        hdf[prefix + "same_hour_2d"] = day_values[2]
        hdf[prefix + "same_hour_3d"] = day_values[3]
        hdf[prefix + "same_hour_7d"] = day_values[7]
        hdf[prefix + "same_hour_14d"] = day_values[14]
        hdf[prefix + "same_hour_21d"] = day_values[21]
        hdf[prefix + "same_hour_28d"] = day_values[28]

        # The averages run over the day lags this horizon can actually observe.
        observable = observable_day_lags(h)
        if observable:
            columns = []
            for d in observable:
                columns.append(day_values[d])
            stacked = np.column_stack(columns)
            hdf[prefix + "mean_same_hour_7d"] = np.nanmean(stacked, axis=1)
            hdf[prefix + "median_same_hour_7d"] = np.nanmedian(stacked, axis=1)
        else:
            hdf[prefix + "mean_same_hour_7d"] = np.nan
            hdf[prefix + "median_same_hour_7d"] = np.nan

        hdf[prefix + "diff_1d"] = hdf[prefix + "same_hour_1d"] - hdf[prefix + "same_hour_2d"]
        hdf[prefix + "diff_7d"] = hdf[prefix + "same_hour_7d"] - hdf[prefix + "same_hour_14d"]

        # Read at the origin, so the whole window lies at or before it.
        hdf[prefix + "roll_24h_mean"] = origin.map(roll_mean).values
        hdf[prefix + "roll_24h_std"] = origin.map(roll_std).values
        hdf[prefix + "roll_24h_max"] = origin.map(roll_max).values
        hdf[prefix + "value_at_origin"] = origin.map(history).values
        hdf[prefix + "prev_day_total"] = origin.map(roll_sum).values
        hdf[prefix + "zero_share_7d"] = origin.map(zero_share).values

        hdf["_target"] = base[target].values
        frames.append(hdf)

    matrix = pd.concat(frames, ignore_index=True).dropna(subset=["_target"])
    if matrix.empty:
        return pd.DataFrame(), pd.Series(dtype=float)
    return matrix[["ts_hour"] + feature_names(target)], matrix["_target"]


# The sparse horizons the models train on; forecasting still covers 1..48.
TRAIN_HORIZONS = [1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 21, 24, 30, 36, 42, 48]
print("training horizons:", TRAIN_HORIZONS, f"({len(TRAIN_HORIZONS)} of "
      f"{cfg.forecast_horizon})")

The pipeline computes the same lags for the ten history features it carries; the two
implementations must agree, so we check them against each other on one device rather than
trust that they do.

In [ ]:
from celine.forecasting.models.lightgbm.features import prepare_training_data

# Cross-check on the biggest exporter of the mature group.
dev_stats = PT.groupby("device_id").agg(
    mean_export=("grid_export", "mean"),
    mean_import=("grid_import", "mean"),
    first_ts=("ts_hour", "min"),
    last_ts=("ts_hour", "max"),
    rows=("ts_hour", "size"),
)
dev_stats["span_days"] = (dev_stats["last_ts"] - dev_stats["first_ts"]).dt.total_seconds() / 86400
mature = dev_stats[dev_stats["span_days"] > 300]
PV_DEV = mature["mean_export"].idxmax()
no_pv = dev_stats.loc[[d for d in dev_stats.index if not HAS_PV[d]]]
IMP_DEV = no_pv["mean_import"].idxmax()

check_dev = PT[PT["device_id"] == PV_DEV].copy()
check_end = check_dev["ts_hour"].max() - pd.Timedelta(days=14)

ours, _ = build_feature_matrix(check_dev, "grid_export", TRAIN_HORIZONS, check_end)
theirs, _ = prepare_training_data(check_dev, "grid_export", check_end, cfg,
                                  horizons=TRAIN_HORIZONS, has_pv=True,
                                  available_columns=set(PT.columns))
ours_sorted = ours.sort_values(["horizon", "ts_hour"]).reset_index(drop=True)
theirs_sorted = theirs.sort_values(["horizon", "ts_hour"]).reset_index(drop=True)
print("rows: ours", len(ours_sorted), "| pipeline", len(theirs_sorted))

rows = []
for name in cfg.features["lags"]:
    col = f"grid_export_{name}"
    mine = ours_sorted[col].to_numpy(dtype=float)
    theirs_values = theirs_sorted[col].to_numpy(dtype=float)
    same = bool(np.allclose(mine, theirs_values, equal_nan=True, rtol=1e-12, atol=1e-12))
    rows.append({"feature": name, "identical": same,
                 "max_abs_diff": float(np.nanmax(np.abs(mine - theirs_values)))})
    assert same, f"{col} differs from the pipeline"
print(pd.DataFrame(rows))
print("\nAll ten shared history features are numerically identical on device",
      f"...{PV_DEV[-5:]}.")

### 4.5 A leak test on a synthetic ramp

The observability rule is an argument; this is the evidence. On a ramp `y(t) = t` every
hour has its own value, so the lag a feature actually read can be recovered by arithmetic:
`implied lag = y(t) - feature`. We build the matrix on such a ramp at four horizons and
require every history feature's implied lag to be at least `h`.

In [ ]:
# A ramp long enough for the 28-day lag plus a month of margin.
ramp_index = pd.date_range("2026-01-01", periods=24 * 70, freq="h", tz="UTC")
ramp = pd.DataFrame({"ts_hour": ramp_index, "device_id": "SYNTHETIC",
                     "grid_export": np.arange(len(ramp_index), dtype=float),
                     "grid_import": 0.0})
# The matrix reads calendar and weather columns off the frame, so give it neutral ones.
for col in BASE_CALENDAR + WEATHER_FROM_FRAME:
    ramp[col] = 0.0

# The lag each feature is supposed to read, in hours. None means horizon-dependent.
expected_lag = {
    "same_hour_1d": None, "same_hour_2d": 48, "same_hour_3d": 72, "same_hour_7d": 168,
    "same_hour_14d": 336, "same_hour_21d": 504, "same_hour_28d": 672,
    "mean_same_hour_7d": None, "median_same_hour_7d": None,
    "roll_24h_mean": None, "roll_24h_max": None, "value_at_origin": None,
}

checks = []
for h in (6, 24, 36, 48):
    Xr, _ = build_feature_matrix(ramp, "grid_export", [h], ramp_index[-1])
    row = Xr[Xr["ts_hour"] == ramp_index[-1]].iloc[0]
    t_value = float(ramp["grid_export"].iloc[-1])
    observable = observable_day_lags(h)

    # What each feature should read, given the horizon.
    if h <= 24:
        want_1d = 24
    else:
        want_1d = 48
    mean_lag = float(np.mean(observable)) * 24
    median_lag = float(np.median(observable)) * 24
    # The 24 h window ending at the origin has mean (origin + origin - 23) / 2.
    window_mean_lag = h + 11.5
    wanted = dict(expected_lag)
    wanted["same_hour_1d"] = want_1d
    wanted["mean_same_hour_7d"] = mean_lag
    wanted["median_same_hour_7d"] = median_lag
    wanted["roll_24h_mean"] = window_mean_lag
    wanted["roll_24h_max"] = h
    wanted["value_at_origin"] = h

    for name, want in wanted.items():
        got = t_value - float(row[f"grid_export_{name}"])
        checks.append({"horizon_h": h, "feature": name, "implied_lag_h": round(got, 1),
                       "expected_lag_h": round(want, 1),
                       "observable (lag >= h)": "yes" if got >= h - 1e-9 else "NO"})
    checks.append({"horizon_h": h, "feature": "diff_1d",
                   "implied_lag_h": float(row["grid_export_diff_1d"]),
                   "expected_lag_h": np.nan,
                   "observable (lag >= h)": "value, not a lag"})

ramp_table = pd.DataFrame(checks)
print("implied lag of every history feature on the ramp y(t) = t:\n")
print(ramp_table.to_string(index=False))

is_lag_row = ramp_table["expected_lag_h"].notna()
assert np.allclose(ramp_table.loc[is_lag_row, "implied_lag_h"],
                   ramp_table.loc[is_lag_row, "expected_lag_h"], atol=1e-6), \
    "a history feature does not read the hour it should"
assert (ramp_table.loc[is_lag_row, "observable (lag >= h)"] == "yes").all(), \
    "a history feature reads after the origin"
print("\nEvery history feature reads at or before the origin at every horizon tested.")

long_band = ramp_table[(ramp_table["horizon_h"] > 24) & (ramp_table["feature"] == "diff_1d")]
print("diff_1d beyond 24 h:", set(long_band["implied_lag_h"]),
      "- same_hour_1d and same_hour_2d are the same column there.")

### 4.6 A look at the matrix

Two devices, chosen programmatically so the notebook is reproducible: the highest mean
export among devices with more than 300 days of span, and the highest mean import among the
devices without PV. Both are frozen 14 days before their own last hour, which is the
holdout of section 5.

In [ ]:
print("PV / grid_export device:", PV_DEV)
print(dev_stats.loc[[PV_DEV]].round(3))
print("\nconsumption-only / grid_import device:", IMP_DEV)
print(dev_stats.loc[[IMP_DEV]].round(3))

df_pv = PT[PT["device_id"] == PV_DEV].copy()
df_imp = PT[PT["device_id"] == IMP_DEV].copy()
TRAIN_END_PV = df_pv["ts_hour"].max() - pd.Timedelta(days=14)
TRAIN_END_IMP = df_imp["ts_hour"].max() - pd.Timedelta(days=14)

t0 = time.time()
X_pv, y_pv = build_feature_matrix(df_pv, "grid_export", TRAIN_HORIZONS, TRAIN_END_PV)
X_imp, y_imp = build_feature_matrix(df_imp, "grid_import", TRAIN_HORIZONS, TRAIN_END_IMP)
print(f"\nboth matrices built in {time.time() - t0:.1f}s")

feats_pv = feature_names("grid_export")
feats_imp = feature_names("grid_import")
print(f"\ngrid_export  X {X_pv.shape} (ts_hour + {len(feats_pv)} features), "
      f"y mean {y_pv.mean():.3f} kWh/h, zeros {100 * (y_pv == 0).mean():.1f}%")
print("train_end:", TRAIN_END_PV)
print(f"grid_import  X {X_imp.shape} (ts_hour + {len(feats_imp)} features), "
      f"y mean {y_imp.mean():.3f} kWh/h, zeros {100 * (y_imp == 0).mean():.1f}%")
print("train_end:", TRAIN_END_IMP)

print("\nrows per horizon for the PV device (the expansion is exact - every horizon sees")
print("every target hour):")
print(X_pv["horizon"].value_counts().sort_index().rename("rows").to_frame().T)

In [ ]:
nan_share = X_pv[feats_pv].isna().mean().mul(100).sort_values(ascending=False)
nan_share = nan_share[nan_share > 0]

fig, ax = plt.subplots(figsize=(10, 5))
y = np.arange(len(nan_share))
colors = []
for feature_name in nan_share.index:
    if FEATURE_FAMILY[feature_name] == "history":
        colors.append("tab:orange")
    else:
        colors.append("tab:blue")
ax.barh(y, nan_share.values, color=colors)
ax.set_yticks(y)
ax.set_yticklabels(nan_share.index)
ax.invert_yaxis()
handles = [mpl.patches.Patch(color="tab:orange", label="target history"),
           mpl.patches.Patch(color="tab:blue", label="calendar / weather / horizon")]
ax.legend(handles=handles)
ax.set_title(f"Features with missing values - grid_export matrix, ...{PV_DEV[-5:]}")
ax.set_xlabel("rows with NaN (%)")
plt.tight_layout()
plt.show()

complete = len(feats_pv) - len(nan_share)
print(f"{complete} of {len(feats_pv)} features have no missing value at all.")
print("The rest are history lags, NaN at the start of the series and inside long gaps:")
print("a 28-day lag has nothing to read for the device's first 28 days. LightGBM routes")
print("NaN down its own branch at every split, so nothing is imputed.")

In [ ]:
# Correlation of every feature with the target, for the PV device.
corr_cols = []
for feature_name in feats_pv:
    if X_pv[feature_name].notna().any() and X_pv[feature_name].std() > 0:
        corr_cols.append(feature_name)
corr_df = X_pv[corr_cols].copy()
corr_df["grid_export"] = y_pv.values
cmat = corr_df.corr(numeric_only=True)

target_corr = cmat["grid_export"].drop("grid_export")
order = target_corr.abs().sort_values(ascending=False).index.tolist()
cmat = cmat.loc[order + ["grid_export"], order + ["grid_export"]]

fig, ax = plt.subplots(figsize=(13, 11))
sns.heatmap(cmat, ax=ax, cmap="coolwarm", center=0, vmin=-1, vmax=1, square=True,
            cbar_kws={"label": "Pearson correlation"})
ax.set_title("Feature correlation, ordered by |correlation| with the target "
             "(last row/column)")
plt.tight_layout()
plt.show()

top = target_corr.loc[target_corr.abs().sort_values(ascending=False).index]
print("strongest correlations with grid_export:")
print(top.head(12).round(3).to_frame("corr_with_target").T)
print("\nweakest:")
print(top.tail(8).round(3).to_frame("corr_with_target").T)

In [ ]:
# The same view for the consumption-only device, as a table rather than a heatmap.
imp_cols = []
for feature_name in feats_imp:
    if X_imp[feature_name].notna().any() and X_imp[feature_name].std() > 0:
        imp_cols.append(feature_name)
imp_input = X_imp[imp_cols].copy()
imp_input["grid_import"] = y_imp.values
imp_corr = imp_input.corr(numeric_only=True)["grid_import"].drop("grid_import")
imp_corr = imp_corr.loc[imp_corr.abs().sort_values(ascending=False).index]

print(f"consumption-only device ...{IMP_DEV[-5:]}, {len(X_imp):,} rows")
print("\nstrongest correlations with grid_import:")
print(imp_corr.head(12).round(3).to_frame("corr_with_target").T)
print("\nweakest:")
print(imp_corr.tail(8).round(3).to_frame("corr_with_target").T)

constant_cols = []
for feature_name in feats_imp:
    if feature_name not in imp_cols:
        constant_cols.append(feature_name)
print("\nfeatures that are constant or empty in this matrix:", constant_cols)

## 5. Which features are useful?

### 5.1 The protocol

**The holdout.** For every device the **last 14 days of its own series** are held out. No
model ever trains on them. Inside that window we take **7 daily forecast origins** — the
first seven midnights after the training cutoff — and from each origin forecast the next
**48 hours**. That is 7 × 48 = 336 scored hours per device-target, spread over a week of
different weather and a whole range of forecast distances.

The score is **MAE in kWh per hour** against the truth, reported per target and per horizon
range (1–8 h, 9–24 h, 25–48 h, and all 48). Two naive baselines run on exactly the same
origins and hours: **naive-168h**, the same hour last week, and **naive-24h**, the same
hour yesterday. Skill is `1 − MAE_model / MAE_naive168`, reported only where the naive MAE
is above 0.01 kWh/h, because a ratio against a near-zero denominator says nothing.

**Why this cannot leak.** The holdout feature rows are built with the same
`build_feature_matrix`, on a window ending at `origin + 48 h`, keeping the single row whose
`ts_hour` is `origin + h`. Every history feature obeys `L >= h`, so no value after the
origin can reach into them; the ramp test of 4.5 is the proof. Weather and calendar are read
at the target hour on purpose — that is the perfect-forecast assumption of 4.2.

**The models.** One LightGBM booster per (device, target), trained on the whole pool of 51
features over the sparse horizons `{1, 2, 3, 4, 5, 6, 8, 10, 12, 15, 18, 21, 24, 30, 36,
42, 48}`. The 48-hour forecast still scores every horizon, because `horizon` is a feature.

Parameters are `cfg.lgb_params` with `learning_rate = 0.1` and `num_boost_round = 300`, and
early stopping after 30 rounds without improvement on the last 15 % of the training rows in
time order (`validation_split_fraction`). This is a **screening setting**: it is chosen so
that 52 fits finish in minutes, not so that any single model is as good as it could be. The
question here is which features help, and that ordering is stable well before the last
boosting round.

**The objective.** `regression` for `grid_export`. For `grid_import`, **Tweedie** with
variance power 1.5, from `lgb_params_by_target`: notebook 01 found the import series about
30 % exact zeros and the export series about 75 %, and after the noise floor of 2.2 the
import series is more zero-inflated still. An L2 objective chases the rare spikes in a
mostly-zero series and loses to the naive baselines; Tweedie is built for non-negative
zero-inflated targets and, unlike an L1 objective, supports monotone constraints.

**The physics priors.** LightGBM is free to fit any shape, including "more irradiance, less
export" when a spurious correlation makes it locally profitable. Monotone constraints forbid
that. They cost training freedom and buy extrapolation safety on weather values the training
window never contained.

**The models we fit.** `grid_export` for the devices with PV, `grid_import` for all fifteen.

In [ ]:
import lightgbm as lgb

from celine.forecasting.core.baselines import naive_forecast

FORECAST_H = cfg.forecast_horizon        # 48
HOLDOUT_DAYS = 14
N_ORIGINS = 7
LOOKBACK_DAYS = 45                       # enough history for the 28-day lag
SEED = cfg.random_seed
NUM_BOOST_ROUND = 300                    # screening setting, see above
LEARNING_RATE = 0.1
EARLY_STOPPING = int(cfg.raw["early_stopping_rounds"])
VALID_FRACTION = float(cfg.raw["validation_split_fraction"])

# Physics priors, stated here rather than read from a file, because they are a modelling
# decision of this notebook.
MONOTONIC = {
    "grid_export": {
        "positive": ["global_tilted_irradiance", "shortwave_radiation",
                     "effective_solar_pv", "clearsky_index", "solar_elevation"],
        "negative": ["cloud_cover"],
    },
    "grid_import_pv": {
        # With panels, import is what the sun did not cover: more sun, less import.
        "positive": ["heating_degree", "cloud_cover"],
        "negative": ["effective_solar_pv", "clearsky_index", "solar_elevation"],
    },
    "grid_import_no_pv": {
        # Without panels the sun says nothing about import; only heating does.
        "positive": ["heating_degree"],
        "negative": [],
    },
}


def monotonic_key(target, has_pv):
    """Which constraint set a (target, has_pv) combination uses."""
    if target == "grid_export":
        return "grid_export"
    if has_pv:
        return "grid_import_pv"
    return "grid_import_no_pv"


def monotonic_vector(features, target, has_pv):
    """The {-1, 0, +1} vector aligned to `features`."""
    rules = MONOTONIC[monotonic_key(target, has_pv)]
    vector = []
    for feature_name in features:
        if feature_name in rules["positive"]:
            vector.append(1)
        elif feature_name in rules["negative"]:
            vector.append(-1)
        else:
            vector.append(0)
    return vector


def lgb_params_for(target):
    """Base parameters plus the per-target objective and the screening settings."""
    params = dict(cfg.lgb_params)
    overrides = cfg.raw.get("lgb_params_by_target", {})
    if target in overrides:
        params.update(overrides[target])
    params["learning_rate"] = LEARNING_RATE
    params["seed"] = SEED
    params["bagging_seed"] = SEED
    params["feature_fraction_seed"] = SEED
    params["verbose"] = -1
    return params


def fit_booster(X, y, features, target, has_pv):
    """Fit one booster, early-stopping on the most recent training rows."""
    order = np.argsort(X["ts_hour"].to_numpy(), kind="stable")
    X_sorted = X.iloc[order]
    y_sorted = y.iloc[order]
    params = lgb_params_for(target)
    # Tweedie needs a non-zero label sum; a device whose whole training window sits under
    # the noise floor has none, so fall back to the plain objective for that one model.
    if params.get("objective") == "tweedie" and float(y_sorted.sum()) <= 0:
        params["objective"] = "regression"
        params["metric"] = "rmse"
        print("    all-zero training target: tweedie falls back to regression")
    params["monotone_constraints"] = monotonic_vector(features, target, has_pv)
    cut = int(len(X_sorted) * VALID_FRACTION)
    train_set = lgb.Dataset(X_sorted[features].iloc[:cut], y_sorted.iloc[:cut],
                            free_raw_data=False)
    valid_set = lgb.Dataset(X_sorted[features].iloc[cut:], y_sorted.iloc[cut:],
                            reference=train_set, free_raw_data=False)
    return lgb.train(params, train_set, num_boost_round=NUM_BOOST_ROUND,
                     valid_sets=[valid_set],
                     callbacks=[lgb.early_stopping(EARLY_STOPPING, verbose=False)])


def holdout_matrix(df_device, target, origin):
    """The 48 feature rows of one forecast origin, one per horizon."""
    window_end = origin + pd.Timedelta(hours=FORECAST_H)
    after_start = df_device["ts_hour"] > origin - pd.Timedelta(days=LOOKBACK_DAYS)
    before_end = df_device["ts_hour"] <= window_end
    window = df_device[after_start & before_end]
    X, _ = build_feature_matrix(window, target, list(range(1, FORECAST_H + 1)), window_end)
    if len(X) == 0:
        return X
    # Keep the one row per horizon whose target hour is origin + h.
    keep = X["ts_hour"] == origin + pd.to_timedelta(X["horizon"], unit="h")
    return X[keep].sort_values("horizon").reset_index(drop=True)


def horizon_range(h):
    """The reporting bucket of a forecast horizon."""
    if h <= 8:
        return "1-8 h"
    if h <= 24:
        return "9-24 h"
    return "25-48 h"


RANGE_ORDER = ["1-8 h", "9-24 h", "25-48 h", "all"]

# grid_export for the devices with PV, grid_import for all of them.
DEVICE_TARGETS = []
for device_id in sorted(HAS_PV):
    if HAS_PV[device_id]:
        DEVICE_TARGETS.append((device_id, "grid_export"))
for device_id in sorted(HAS_PV):
    DEVICE_TARGETS.append((device_id, "grid_import"))

print(f"{len(DEVICE_TARGETS)} models: "
      f"{sum(1 for _, t in DEVICE_TARGETS if t == 'grid_export')} grid_export + "
      f"{sum(1 for _, t in DEVICE_TARGETS if t == 'grid_import')} grid_import")
print("seed:", SEED, "| rounds:", NUM_BOOST_ROUND, "| learning rate:", LEARNING_RATE,
      "| early stopping:", EARLY_STOPPING)

### 5.2 Run it

Train the models, forecast the seven origins of each one, score against the truth and the
two naive baselines.

In [ ]:
t_train = time.time()
BOOSTERS = {}
HOLDOUT_X = {}
TRAIN_END = {}
rows = []

for device_id, target in DEVICE_TARGETS:
    df_device = PT[PT["device_id"] == device_id].copy()
    has_pv = bool(HAS_PV[device_id])
    features = feature_names(target)
    last_hour = df_device["ts_hour"].max()
    train_end = last_hour - pd.Timedelta(days=HOLDOUT_DAYS)
    TRAIN_END[(device_id, target)] = train_end

    X, y = build_feature_matrix(df_device, target, TRAIN_HORIZONS, train_end)
    booster = fit_booster(X, y, features, target, has_pv)
    BOOSTERS[(device_id, target)] = booster

    # The truth, indexed by hour, for scoring.
    actual = df_device.set_index("ts_hour")[target]
    actual = actual[~actual.index.duplicated(keep="last")]

    for k in range(N_ORIGINS):
        origin = train_end.ceil("D") + pd.Timedelta(days=k)
        if origin + pd.Timedelta(hours=FORECAST_H) > last_hour:
            continue
        X_hold = holdout_matrix(df_device, target, origin)
        if len(X_hold) == 0:
            continue
        HOLDOUT_X[(device_id, target, origin)] = X_hold
        prediction = np.maximum(0.0, booster.predict(X_hold[features]))
        naive168 = naive_forecast(df_device, target, origin, cfg,
                                  lag_hours=168).set_index("horizon")["prediction"]
        naive24 = naive_forecast(df_device, target, origin, cfg,
                                 lag_hours=24).set_index("horizon")["prediction"]
        for i in range(len(X_hold)):
            h = int(X_hold["horizon"].iloc[i])
            ts = X_hold["ts_hour"].iloc[i]
            rows.append({"device_id": device_id, "target": target, "has_pv": has_pv,
                         "origin": origin, "horizon": h, "ts_hour": ts,
                         "actual": actual.get(ts, np.nan),
                         "model": prediction[i],
                         "naive168": naive168.get(h, np.nan),
                         "naive24": naive24.get(h, np.nan)})
    print(f"  ...{device_id[-5:]} {target}: {len(X):,} training rows, "
          f"{booster.num_trees()} trees  [{time.time() - t_train:.0f}s]")

RES = pd.DataFrame(rows).dropna(subset=["actual"]).reset_index(drop=True)
RES["hrange"] = RES["horizon"].map(horizon_range)
TRAIN_SECONDS = time.time() - t_train
print(f"\n{len(BOOSTERS)} models trained and scored in {TRAIN_SECONDS:.0f}s")
print(f"holdout rows: {len(RES):,} "
      f"({RES['origin'].nunique()} distinct origins, "
      f"{RES.groupby(['device_id', 'target'])['origin'].nunique().mean():.1f} per model)")
print("origins:", RES["origin"].min(), "->", RES["origin"].max())

In [ ]:
def mae_table(frame, columns):
    """MAE per horizon range, plus a row over all horizons."""
    out = []
    for name in RANGE_ORDER:
        if name == "all":
            sub = frame
        else:
            sub = frame[frame["hrange"] == name]
        row = {"horizon_range": name, "rows": len(sub)}
        for col in columns:
            row["MAE_" + col] = round(float((sub[col] - sub["actual"]).abs().mean()), 4)
        out.append(row)
    table = pd.DataFrame(out).set_index("horizon_range")
    # Skill only where the reference error is big enough for a ratio to mean anything.
    skill = []
    for name in RANGE_ORDER:
        reference = table.loc[name, "MAE_naive168"]
        if reference > 0.01:
            skill.append(round(1 - table.loc[name, "MAE_model"] / reference, 4))
        else:
            skill.append(np.nan)
    table["skill_vs_naive168"] = skill
    return table


MAE_FULL = {}
for target in cfg.targets:
    sub = RES[RES["target"] == target]
    table = mae_table(sub, ["model", "naive168", "naive24"])
    MAE_FULL[target] = table
    print(f"=== {target} - {sub['device_id'].nunique()} devices, "
          f"mean actual {sub['actual'].mean():.3f} kWh/h ===")
    print(table)
    print()

In [ ]:
# One row per device-target: how the model did against the week-ago baseline.
per_device_rows = []
for (device_id, target), gd in RES.groupby(["device_id", "target"]):
    mae_model = float((gd["model"] - gd["actual"]).abs().mean())
    mae_n168 = float((gd["naive168"] - gd["actual"]).abs().mean())
    mae_n24 = float((gd["naive24"] - gd["actual"]).abs().mean())
    if mae_n168 > 0.01:
        skill = 1 - mae_model / mae_n168
    else:
        skill = np.nan
    per_device_rows.append({
        "device": device_id[-5:], "target": target, "has_pv": bool(HAS_PV[device_id]),
        "rows": len(gd), "mean_actual": round(float(gd["actual"].mean()), 3),
        "MAE_model": round(mae_model, 4), "MAE_naive168": round(mae_n168, 4),
        "MAE_naive24": round(mae_n24, 4),
        "skill": round(skill, 3) if skill == skill else np.nan,
    })
PER_DEVICE = pd.DataFrame(per_device_rows).sort_values(["target", "skill"],
                                                       ascending=[True, False])
beats = PER_DEVICE["MAE_model"] < PER_DEVICE["MAE_naive168"]
print(f"the model beats naive-168h on {int(beats.sum())} of {len(PER_DEVICE)} device-targets")
PER_DEVICE.reset_index(drop=True)

In [ ]:
series = [("model", "LightGBM, full pool", "tab:blue"),
          ("naive168", "naive: same hour 168 h ago", "tab:orange"),
          ("naive24", "naive: same hour 24 h ago", "tab:green")]

for target in cfg.targets:
    sub = RES[RES["target"] == target]
    table = MAE_FULL[target]

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.0))

    # Left: MAE per horizon range.
    ax = axes[0]
    x = np.arange(len(RANGE_ORDER))
    for k, (col, label, color) in enumerate(series):
        values = table.loc[RANGE_ORDER, "MAE_" + col].values
        ax.bar(x + (k - 1) * 0.26, values, width=0.24, color=color, label=label)
        for xi, v in zip(x + (k - 1) * 0.26, values):
            ax.text(xi, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)
    ax.set_xticks(x)
    ax.set_xticklabels(RANGE_ORDER)
    ax.legend()
    ax.set_title(f"{target} - holdout MAE by horizon range")
    ax.set_xlabel("horizon range")
    ax.set_ylabel("MAE (kWh per hour)")

    # Right: the same three series, one point per forecast horizon.
    ax = axes[1]
    horizon_rows = []
    for h, gd in sub.groupby("horizon"):
        horizon_rows.append({
            "horizon": h,
            "model": (gd["model"] - gd["actual"]).abs().mean(),
            "naive168": (gd["naive168"] - gd["actual"]).abs().mean(),
            "naive24": (gd["naive24"] - gd["actual"]).abs().mean(),
        })
    by_h = pd.DataFrame(horizon_rows).set_index("horizon")
    for col, label, color in series:
        ax.plot(by_h.index, by_h[col], color=color, label=label)
    for boundary in (8.5, 24.5):
        ax.axvline(boundary, color="grey", linewidth=1.0)
    ax.legend()
    ax.set_xticks([1, 8, 16, 24, 32, 40, 48])
    ax.set_title("...and horizon by horizon")
    ax.set_xlabel("forecast horizon (hours ahead)")
    ax.set_ylabel("MAE (kWh per hour)")
    plt.tight_layout()
    plt.show()

#### Is the early stopping too aggressive?

The boosters above stop after a few dozen trees, which is little enough to wonder whether
the screening setting is hiding what the models could do. The question has an answer in the
data rather than in intuition, so we ask it: take four device-targets — the two biggest
exporters among the mature meters and the two consumption-only meters with the most import
— and fit each of them three ways on the same training matrix.

* **(a)** the notebook setting: `learning_rate = 0.1`, up to 300 rounds, early stopping 30;
* **(b)** `learning_rate = 0.1`, **300 fixed rounds, no early stopping** — what the model
  looks like if we simply let it run;
* **(c)** `learning_rate = 0.05`, up to 500 rounds, early stopping 50 — a slower, longer
  fit.

All three are scored on the same seven origins as everything else, and we print the
validation curve at 30, 100 and 300 rounds so the stopping point can be seen rather than
trusted.

In [ ]:
def fit_variant(X, y, features, target, has_pv, learning_rate, rounds, early_stop):
    """Fit one booster with a given learning rate, round budget and patience.

    The temporal split is the one fit_booster uses: the last 15 % of the training
    rows in time order are the validation set. Returns the booster, the metric name
    and the whole validation curve, so the stopping point can be inspected instead of
    trusted.
    """
    order = np.argsort(X["ts_hour"].to_numpy(), kind="stable")
    X_sorted = X.iloc[order]
    y_sorted = y.iloc[order]
    params = lgb_params_for(target)
    params["learning_rate"] = learning_rate
    if params.get("objective") == "tweedie" and float(y_sorted.sum()) <= 0:
        params["objective"] = "regression"
        params["metric"] = "rmse"
    params["monotone_constraints"] = monotonic_vector(features, target, has_pv)
    cut = int(len(X_sorted) * VALID_FRACTION)
    train_set = lgb.Dataset(X_sorted[features].iloc[:cut], y_sorted.iloc[:cut],
                            free_raw_data=False)
    valid_set = lgb.Dataset(X_sorted[features].iloc[cut:], y_sorted.iloc[cut:],
                            reference=train_set, free_raw_data=False)
    history = {}
    callbacks = [lgb.record_evaluation(history)]
    if early_stop is not None:
        callbacks.append(lgb.early_stopping(early_stop, verbose=False))
    booster = lgb.train(params, train_set, num_boost_round=rounds,
                        valid_sets=[valid_set], valid_names=["valid"],
                        callbacks=callbacks)
    metric_name = list(history["valid"])[0]
    return booster, metric_name, history["valid"][metric_name]


def holdout_mae_of(booster, device_id, target, features):
    """MAE of one booster over the same seven origins the rest of section 5 uses."""
    scored = RES[(RES["device_id"] == device_id) & (RES["target"] == target)]
    truth = scored.set_index(["origin", "horizon"])["actual"]
    errors = []
    for key in sorted(HOLDOUT_X):
        if key[0] != device_id or key[1] != target:
            continue
        X_hold = HOLDOUT_X[key]
        prediction = np.maximum(0.0, booster.predict(X_hold[features]))
        for i in range(len(X_hold)):
            actual = truth.get((key[2], int(X_hold["horizon"].iloc[i])), np.nan)
            if actual == actual:
                errors.append(abs(prediction[i] - actual))
    if not errors:
        return np.nan
    return float(np.mean(errors))


# The four device-targets, chosen programmatically so the check is reproducible.
mature_devices = dev_stats[dev_stats["span_days"] > 300]
mature_exporters = []
for device_id in mature_devices.index:
    if HAS_PV[device_id]:
        mature_exporters.append(device_id)
top_exporters = mature_devices.loc[mature_exporters, "mean_export"].nlargest(2).index.tolist()
consumption_only = []
for device_id in dev_stats.index:
    if not HAS_PV[device_id]:
        consumption_only.append(device_id)
top_importers = dev_stats.loc[consumption_only, "mean_import"].nlargest(2).index.tolist()

CHECK_TARGETS = []
for device_id in top_exporters:
    CHECK_TARGETS.append((device_id, "grid_export"))
for device_id in top_importers:
    CHECK_TARGETS.append((device_id, "grid_import"))
print("checked device-targets:", [(d[-5:], t) for d, t in CHECK_TARGETS])

VARIANTS = [
    ("a) lr 0.10, <=300 rounds, stop 30", LEARNING_RATE, NUM_BOOST_ROUND, EARLY_STOPPING),
    ("b) lr 0.10, 300 rounds, no stop", 0.10, 300, None),
    ("c) lr 0.05, <=500 rounds, stop 50", 0.05, 500, 50),
]

t_check = time.time()
check_rows = []
for device_id, target in CHECK_TARGETS:
    df_device = PT[PT["device_id"] == device_id].copy()
    has_pv = bool(HAS_PV[device_id])
    features = feature_names(target)
    X, y = build_feature_matrix(df_device, target, TRAIN_HORIZONS,
                                TRAIN_END[(device_id, target)])
    scored = RES[(RES["device_id"] == device_id) & (RES["target"] == target)]
    mae_naive = float((scored["naive168"] - scored["actual"]).abs().mean())
    for label, learning_rate, rounds, early_stop in VARIANTS:
        booster, metric_name, curve = fit_variant(X, y, features, target, has_pv,
                                                  learning_rate, rounds, early_stop)
        row = {"device": device_id[-5:], "target": target, "variant": label,
               "trees": booster.num_trees(),
               "best_iter": booster.best_iteration or len(curve),
               "metric": metric_name}
        # The validation curve at three fixed budgets, where the run got that far.
        for checkpoint in (30, 100, 300):
            if len(curve) >= checkpoint:
                row[f"val@{checkpoint}"] = round(float(curve[checkpoint - 1]), 4)
            else:
                row[f"val@{checkpoint}"] = np.nan
        row["MAE_holdout"] = round(holdout_mae_of(booster, device_id, target, features), 4)
        row["MAE_naive168"] = round(mae_naive, 4)
        check_rows.append(row)
    print(f"  ...{device_id[-5:]} {target}: three variants fitted "
          f"[{time.time() - t_check:.0f}s]")

STOPPING_CHECK = pd.DataFrame(check_rows)
print(f"\nearly-stopping check: {len(check_rows)} fits in {time.time() - t_check:.0f}s")
print(STOPPING_CHECK.to_string(index=False))

# Does letting it run, or slowing it down, actually beat the notebook setting?
pivot = STOPPING_CHECK.pivot_table(index=["device", "target"], columns="variant",
                                   values="MAE_holdout")
baseline = VARIANTS[0][0]
print("\nholdout MAE against the notebook setting (negative = better):")
for label, _, _, _ in VARIANTS[1:]:
    change = (pivot[label] - pivot[baseline]) / pivot[baseline] * 100
    parts = []
    for index, value in change.items():
        parts.append(f"{index[0]} {index[1].split('_')[1]} {value:+.1f}%")
    print(f"  {label}: " + ", ".join(parts))

### 5.3 The verdict, feature by feature

Two measurements per model.

**Gain share.** How much of the total loss reduction of a booster came from splits on this
feature, as a percentage. It says what the model *uses*; a feature with zero gain is dead
weight.

**Permutation importance on the holdout.** We shuffle one feature's column across the
holdout rows, keep everything else as it is, predict again, and record how much the MAE
moved. Three shuffles with a seeded generator, averaged. This is the honest question: if
this feature carried no information, how much worse would the forecast be? It is measured
on rows no model saw.

The two disagree in a useful way. Permutation splits the credit of two redundant features
between them and can score both near zero, because shuffling one leaves the other to carry
the signal. Gain sees them both being used. So we keep a feature when **either** measure
says it matters.

**The rule**, stated once and applied mechanically, on the mean relative ΔMAE across a
target's device-targets:

* **useful** — mean relative ΔMAE ≥ +1 %
* **harmful** — mean relative ΔMAE ≤ −1 %
* **neutral** — anything in between

A feature is **kept** if it is useful **or** its mean gain share is at least 1 %.

In [ ]:
t_perm = time.time()
N_SHUFFLES = 3
perm_rows = []
gain_rows = []

for (device_id, target), booster in BOOSTERS.items():
    features = feature_names(target)

    # All holdout rows of this model, in one frame, with the matching truth.
    origins = []
    for key in HOLDOUT_X:
        if key[0] == device_id and key[1] == target:
            origins.append(key)
    origins = sorted(origins, key=lambda key: key[2])
    X_all = pd.concat([HOLDOUT_X[key] for key in origins], ignore_index=True)

    scored = RES[(RES["device_id"] == device_id) & (RES["target"] == target)]
    truth_by_key = scored.set_index(["origin", "horizon"])["actual"]
    truth = []
    for key in origins:
        for h in HOLDOUT_X[key]["horizon"]:
            truth.append(truth_by_key.get((key[2], int(h)), np.nan))
    truth = np.asarray(truth, dtype=float)
    has_truth = ~np.isnan(truth)

    base_prediction = np.maximum(0.0, booster.predict(X_all[features]))
    base_mae = float(np.abs(base_prediction[has_truth] - truth[has_truth]).mean())

    rng = np.random.default_rng(SEED)
    X_shuffled = X_all[features].copy()
    for feature_name in features:
        column = X_shuffled[feature_name].to_numpy().copy()
        deltas = []
        for _ in range(N_SHUFFLES):
            X_shuffled[feature_name] = column[rng.permutation(len(column))]
            shuffled_prediction = np.maximum(0.0, booster.predict(X_shuffled))
            shuffled_mae = float(
                np.abs(shuffled_prediction[has_truth] - truth[has_truth]).mean())
            deltas.append(shuffled_mae - base_mae)
        X_shuffled[feature_name] = column          # put the real column back
        perm_rows.append({"device_id": device_id, "target": target,
                          "feature": feature_name, "base_mae": base_mae,
                          "delta_mae": float(np.mean(deltas))})

    gains = pd.Series(booster.feature_importance("gain"), index=booster.feature_name())
    total_gain = float(gains.sum())
    for feature_name in features:
        if total_gain > 0:
            share = 100 * float(gains.get(feature_name, 0.0)) / total_gain
        else:
            share = 0.0
        gain_rows.append({"device_id": device_id, "target": target,
                          "feature": feature_name, "gain_pct": share})

PERM = pd.DataFrame(perm_rows)
GAIN = pd.DataFrame(gain_rows)
# Relative, so devices of very different size can be averaged.
PERM["delta_pct"] = 100 * PERM["delta_mae"] / PERM["base_mae"].where(PERM["base_mae"] > 0)
print(f"permutation importance: {len(PERM):,} measurements in {time.time() - t_perm:.0f}s "
      f"({N_SHUFFLES} shuffles per feature per model)")

In [ ]:
USEFUL_THRESHOLD = 1.0      # mean relative delta-MAE, per cent
GAIN_THRESHOLD = 1.0        # mean gain share, per cent

VERDICT = {}
SELECTED_FEATURES = {}
for target in cfg.targets:
    perm_target = PERM[PERM["target"] == target]
    gain_target = GAIN[GAIN["target"] == target]
    summary = perm_target.groupby("feature").agg(
        mean_delta_pct=("delta_pct", "mean"),
        share_models_positive=("delta_pct", lambda s: float((s > 0).mean())),
        models=("delta_pct", "size"),
    )
    summary["mean_gain_pct"] = gain_target.groupby("feature")["gain_pct"].mean()
    summary["family"] = [FEATURE_FAMILY[f] for f in summary.index]

    verdicts = []
    for value in summary["mean_delta_pct"]:
        if value >= USEFUL_THRESHOLD:
            verdicts.append("useful")
        elif value <= -USEFUL_THRESHOLD:
            verdicts.append("harmful")
        else:
            verdicts.append("neutral")
    summary["verdict"] = verdicts
    is_useful = summary["verdict"] == "useful"
    carries_gain = summary["mean_gain_pct"] >= GAIN_THRESHOLD
    summary["kept"] = is_useful | carries_gain
    summary = summary.sort_values("mean_delta_pct", ascending=False)
    VERDICT[target] = summary

    kept = []
    for feature_name in feature_names(target):
        if bool(summary.loc[feature_name, "kept"]):
            kept.append(feature_name)
    SELECTED_FEATURES[target] = kept

    print(f"=== {target} - {int(summary['models'].iloc[0])} device-targets, "
          f"{len(kept)} of {len(summary)} features kept ===")
    view = summary[["family", "mean_delta_pct", "share_models_positive", "mean_gain_pct",
                    "verdict", "kept"]].copy()
    view["mean_delta_pct"] = view["mean_delta_pct"].round(2)
    view["share_models_positive"] = view["share_models_positive"].round(2)
    view["mean_gain_pct"] = view["mean_gain_pct"].round(2)
    print(view.to_string())
    print()

In [ ]:
for target in cfg.targets:
    summary = VERDICT[target]
    fig, ax = plt.subplots(figsize=(10, 12))
    y = np.arange(len(summary))
    colors = []
    for verdict in summary["verdict"]:
        if verdict == "useful":
            colors.append("tab:blue")
        elif verdict == "harmful":
            colors.append("tab:red")
        else:
            colors.append("tab:grey")
    ax.barh(y, summary["mean_delta_pct"].values, color=colors)
    ax.axvline(USEFUL_THRESHOLD, color="black", linestyle="--", linewidth=0.8)
    ax.axvline(-USEFUL_THRESHOLD, color="black", linestyle="--", linewidth=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(summary.index, fontsize=8)
    ax.invert_yaxis()
    handles = [mpl.patches.Patch(color="tab:blue", label="useful (>= +1 %)"),
               mpl.patches.Patch(color="tab:grey", label="neutral"),
               mpl.patches.Patch(color="tab:red", label="harmful (<= -1 %)")]
    ax.legend(handles=handles, loc="lower right")
    ax.set_title(f"{target} - holdout MAE increase when the feature is shuffled")
    ax.set_xlabel("mean change in MAE when the column is shuffled (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
# How much each family carries, per target.
for target in cfg.targets:
    summary = VERDICT[target]
    family = summary.groupby("family").agg(
        features=("mean_delta_pct", "size"),
        kept=("kept", "sum"),
        total_delta_pct=("mean_delta_pct", "sum"),
        total_gain_pct=("mean_gain_pct", "sum"),
    ).round(2)
    print(f"{target} - by family:")
    print(family)
    print()

In [ ]:
# The two lists, printed in full.
for target in cfg.targets:
    summary = VERDICT[target]
    kept = SELECTED_FEATURES[target]
    dropped = []
    for feature_name in feature_names(target):
        if feature_name not in kept:
            dropped.append(feature_name)

    print(f"SELECTED_FEATURES[{target!r}] - {len(kept)} features")
    for feature_name in kept:
        row = summary.loc[feature_name]
        print(f"    {feature_name:<38} {row['family']:<8} "
              f"dMAE {row['mean_delta_pct']:+6.2f}%  gain {row['mean_gain_pct']:5.2f}%  "
              f"{row['verdict']}")
    print(f"\n  dropped - {len(dropped)} features")
    for feature_name in dropped:
        row = summary.loc[feature_name]
        print(f"    {feature_name:<38} {row['family']:<8} "
              f"dMAE {row['mean_delta_pct']:+6.2f}%  gain {row['mean_gain_pct']:5.2f}%  "
              f"{row['verdict']}")
    print()

### 5.4 Confirmation

Dropping a feature is only free if the forecast does not get worse. We retrain all the
models on the selected set alone, score the same origins and the same hours, and compare.

In [ ]:
t_confirm = time.time()
selected_rows = []
for device_id, target in DEVICE_TARGETS:
    df_device = PT[PT["device_id"] == device_id].copy()
    has_pv = bool(HAS_PV[device_id])
    features = SELECTED_FEATURES[target]
    train_end = TRAIN_END[(device_id, target)]

    X, y = build_feature_matrix(df_device, target, TRAIN_HORIZONS, train_end)
    booster = fit_booster(X, y, features, target, has_pv)

    for key in sorted(HOLDOUT_X):
        if key[0] != device_id or key[1] != target:
            continue
        X_hold = HOLDOUT_X[key]
        prediction = np.maximum(0.0, booster.predict(X_hold[features]))
        for i in range(len(X_hold)):
            selected_rows.append({"device_id": device_id, "target": target,
                                  "origin": key[2],
                                  "horizon": int(X_hold["horizon"].iloc[i]),
                                  "selected": prediction[i]})

SELECTED_PRED = pd.DataFrame(selected_rows)
RES = RES.merge(SELECTED_PRED, on=["device_id", "target", "origin", "horizon"], how="left")
print(f"{len(DEVICE_TARGETS)} models retrained on the selected sets in "
      f"{time.time() - t_confirm:.0f}s")

MAE_SELECTED = {}
for target in cfg.targets:
    sub = RES[RES["target"] == target]
    table = mae_table(sub, ["model", "selected", "naive168"])
    table["delta_vs_full"] = (table["MAE_selected"] - table["MAE_model"]).round(4)
    table["improvement_%"] = ((1 - table["MAE_selected"] / table["MAE_model"]) * 100).round(2)
    MAE_SELECTED[target] = table
    print(f"=== {target}: full pool ({len(feature_names(target))} features) vs selected "
          f"({len(SELECTED_FEATURES[target])}) ===")
    print(table)
    print()

In [ ]:
# Per device-target: does the smaller model hold up?
wins = []
for (device_id, target), gd in RES.groupby(["device_id", "target"]):
    mae_full = float((gd["model"] - gd["actual"]).abs().mean())
    mae_sel = float((gd["selected"] - gd["actual"]).abs().mean())
    mae_n168 = float((gd["naive168"] - gd["actual"]).abs().mean())
    wins.append({"device": device_id[-5:], "target": target,
                 "MAE_full": round(mae_full, 4), "MAE_selected": round(mae_sel, 4),
                 "MAE_naive168": round(mae_n168, 4),
                 "selected_better": mae_sel <= mae_full,
                 "delta": round(mae_sel - mae_full, 4)})
WINS = pd.DataFrame(wins).sort_values(["target", "delta"])
print(f"the selected set is at least as good on {int(WINS['selected_better'].sum())} "
      f"of {len(WINS)} device-targets")
for target in cfg.targets:
    sub = WINS[WINS["target"] == target]
    print(f"  {target}: {int(sub['selected_better'].sum())} of {len(sub)}, "
          f"mean delta {sub['delta'].mean():+.4f} kWh/h")
WINS.reset_index(drop=True)

## 6. Saved outputs

Everything this notebook produced, into `notebooks/data/` (gitignored):

* `processed_hourly.parquet` — the cleaned hourly frame, cohort only, trimmed per device;
* `features_example_grid_export.parquet` / `features_example_grid_import.parquet` — the two
  training matrices of 4.6 with their target column appended, so the exact rows a model saw
  can be inspected;
* `device_quality.csv` — written in section 3;
* `holdout_results.csv` — one row per origin × horizon × device-target, with the truth, the
  full-pool forecast, the selected-set forecast and the two naive baselines;
* `feature_verdict.csv` — the verdict table of 5.3 for both targets;
* `selected_features.json` — the two selected lists.

In [ ]:
out_paths = {
    "processed_hourly": DATA_DIR / "processed_hourly.parquet",
    "features_example_grid_export": DATA_DIR / "features_example_grid_export.parquet",
    "features_example_grid_import": DATA_DIR / "features_example_grid_import.parquet",
    "device_quality": DATA_DIR / "device_quality.csv",
    "holdout_results": DATA_DIR / "holdout_results.csv",
    "feature_verdict": DATA_DIR / "feature_verdict.csv",
    "selected_features": DATA_DIR / "selected_features.json",
}

PT.to_parquet(out_paths["processed_hourly"], index=False)

features_export = X_pv.copy()
features_export["grid_export"] = y_pv.values
features_export.to_parquet(out_paths["features_example_grid_export"], index=False)
features_import = X_imp.copy()
features_import["grid_import"] = y_imp.values
features_import.to_parquet(out_paths["features_example_grid_import"], index=False)

RES.to_csv(out_paths["holdout_results"], index=False)

verdict_frames = []
for target in cfg.targets:
    frame = VERDICT[target].copy()
    frame.insert(0, "target", target)
    frame.index.name = "feature"
    verdict_frames.append(frame.reset_index())
VERDICT_ALL = pd.concat(verdict_frames, ignore_index=True)
VERDICT_ALL.to_csv(out_paths["feature_verdict"], index=False)

with open(out_paths["selected_features"], "w", encoding="utf-8") as handle:
    json.dump(SELECTED_FEATURES, handle, indent=2)

frames = {
    "processed_hourly": PT,
    "features_example_grid_export": features_export,
    "features_example_grid_import": features_import,
    "device_quality": q,
    "holdout_results": RES,
    "feature_verdict": VERDICT_ALL,
    "selected_features": pd.DataFrame(
        [{"target": t, "features": len(f)} for t, f in SELECTED_FEATURES.items()]),
}
summary_rows = []
for key, path in out_paths.items():
    frame = frames[key]
    summary_rows.append({"file": path.name, "rows": len(frame), "cols": frame.shape[1],
                         "size_MB": round(path.stat().st_size / 1e6, 3)})
print(pd.DataFrame(summary_rows))
print("\ndevices in processed_hourly:", PT["device_id"].nunique())
print("span:", PT["ts_hour"].min(), "->", PT["ts_hour"].max())
print(f"\nnotebook runtime so far: {(time.time() - NB_START) / 60:.1f} min")

## 7. Findings

Written after executing the notebook. Every number below is from this run.

### What the cleaning does to the cohort

1. **372,683 fifteen-minute readings become 94,540 hourly rows.** That is 61.8 % of the
   41-device extract, spread over 2025-08-26 09:30 → 2026-09-22 13:45 UTC. The raw/hourly
   ratio is **3.94**, so the data is dense. **4,050 hours (4.28 %) are partial** and get
   scaled by `4 / n_quarters`: 2,950 are missing one quarter, 773 are missing two, and
   **327 hours are a single quarter multiplied by four**. That last group is an
   extrapolation with an error bar four times the reading's own, and nothing downstream can
   tell those hours apart — `partial_hour` survives into the processed frame but nothing
   reads it.

2. **The cohort onboards in four waves and three meters stop early.** Eight came online on
   2025-08-26, one on 2025-09-15, one on 2026-02-03 and five on 2026-05-21; spans run from
   124 to 392 days, median 338. The five meters of the 2026-05-21 wave will gain their
   earlier readings from the `mySET` table once notebook 01 reads it, so their short spans
   are a limitation of this run rather than of the meters. `...BC3F0` last reported on 2026-06-18, `...CD3D0` on
   2026-08-19 and `...62FD4` on 2026-09-09. Because the holdout of section 5 is each
   device's *own* last 14 days, a stalled meter is still scored — on its own last week,
   not on the most recent one. That is why the 26 models span 28 distinct origins between
   2026-06-05 and 2026-09-15 rather than sharing seven.

3. **The two DST nights behave exactly as predicted.** On the fall-back night the UTC hour
   **2025-10-26 00:00 is entirely absent** for all nine meters then online — the twice-run
   local hour 02:00 collapses onto one UTC instant — which is a one-hour hole the grid
   fills. On the spring-forward night **there is no hole at all**: the ten meters then
   online deliver 40 readings in every UTC hour of the window, and the weather frame has
   all ten hours. The local clock does skip 02:00 (the printed local hours jump from
   01:00 CET to 03:00 CEST), but UTC runs straight through it. The asymmetry is worth
   remembering: autumn costs an hour, spring costs nothing.

4. **The 0.020 kWh/h noise floor is not a rounding detail.** It zeroes **13,165 import
   values (13.9 % of all hours)** and 8,059 export values (8.5 %), pushing the exact-zero
   share from 34.2 % to 48.2 % for import and from 61.7 % to 70.2 % for export. The energy
   erased is negligible — 90.9 kWh of 23,397 imported (0.39 %) and 26.9 kWh of 58,691
   exported (0.05 %) — which is the point: the floor removes noise, not energy. But it
   makes `grid_import` substantially more zero-inflated than the meter is, which is the
   reason the import models are fit with a Tweedie objective.

5. **The regular grid is mostly padding, and the trim is nearly free.** 15 devices ×
   9,413 hours = 141,195 rows, of which 46,655 did not exist before and only **275 are
   interpolated**. Of the 46,380 flagged hours, **78.8 % are leading** — hours before the
   device's first reading — 7.4 % are trailing and only **13.8 % (6,392) are genuine
   internal gaps**, spread over 191 runs on all 15 devices (median 2 h, 99th percentile
   590 h, worst 1,493 h). Keeping each device on its own first→last hour drops 28.3 % of
   the rows and takes the gap share from **32.8 % to 6.3 %** with no data lost. Coverage
   inside a device's own span is good for most meters but not all: `...8AA78` sits at
   **62.5 %**, `...2E6EC` at 80.8 % and `...62FD4` at 81.6 %, while the rest are above
   88 %.

6. **The calendar features check out.** `hour_local` is genuinely local: the cohort's mean
   export peaks in the **12:00–13:00 local bin**, inside the window expected for a
   south-facing site at this longitude, and the cyclic encoding traces a clean circle over
   a week.

7. **The outlier flags fire on ordinary daylight, not on bad readings.** 846 export flags
   (0.89 % of observed hours, 11 devices) and 1,207 import flags (1.27 %, 13 devices). The
   flagged export values have a median of **2.556 kWh/h against an overall median of
   0.000**, so the flag mostly marks ordinary daytime production: the centred 168 h rolling
   standard deviation is computed across a mostly-zero series and collapses at the
   shoulders of the solar day. They are informational only, and the window being centred
   makes them retrospective by construction.

8. **The step-by-step walk and the one-call chain agree exactly.** `assert_frame_equal`
   passes on 28 numeric and 3 boolean columns over the 101,207 shared rows, so everything
   measured above describes the chain the command-line pipeline runs.

### What a one-hour fill costs

9. **Linear interpolation over one hour is cheap, and it beats the obvious alternative.**
   Measured on 6,000 real hours per target that we hid and refilled: the linear mid-point
   is off by **0.132 kWh/h on export** (mean value 0.646) and **0.073 on import** (mean
   value 0.385), against **0.246** and **0.097** for a carry-forward fill — **46 % and
   25 % better**. The carry-forward error is also the plain hour-to-hour variability of the
   series, so the fill lands comfortably inside the noise the series already has. And it is
   applied to **275 rows out of 141,195**, two hundredths of a per cent, while the
   alternative — leaving the hour missing — would strike out lag values for up to 28 days
   of downstream rows.

### The PV split

10. **Eleven of the fifteen meters export, four never do** (`...BC3F0`, `...89CF4`,
    `...89ED4`, `...2E6EC`), on a mean-export threshold of 0.01 kWh/h. The flag matters
    because it changes the physics of the import model, not just which targets exist: with
    panels, import is what the sun did not cover, so irradiance enters with a *negative*
    monotone constraint; without them only heating degree-days are constrained at all.
    Two of the exporting meters go the other way and import essentially nothing —
    `...C9968` has a mean import of **exactly 0.0000 kWh/h**, not "small": every import
    value it reports sits under the noise floor and is snapped to zero. Its import model is
    degenerate by construction; the Tweedie objective cannot be fit on an all-zero label
    vector at all, so that one model falls back to the plain L2 objective and produces a
    single tree.

### Does any of it work?

11. **Yes for export, weakly for import**, on a 14-day / 7-origin holdout per device
    (48 h forecasts, 8,712 scored hours, no model trained on any of them):

    | target | horizon range | MAE model | MAE naive 168 h | MAE naive 24 h | skill |
    |---|---|---|---|---|---|
    | grid_export | 1–8 h | **0.1884** | 0.2652 | 0.2248 | +29.0 % |
    | grid_export | 9–24 h | **0.3715** | 0.6923 | 0.4355 | +46.3 % |
    | grid_export | 25–48 h | **0.3129** | 0.5228 | 0.3291 | +40.2 % |
    | grid_export | all (1–48 h) | **0.3116** | 0.5361 | 0.3471 | **+41.9 %** |
    | grid_import | 1–8 h | **0.0726** | 0.0783 | 0.0776 | +7.3 % |
    | grid_import | 9–24 h | **0.1347** | 0.1540 | 0.1798 | +12.5 % |
    | grid_import | 25–48 h | **0.1224** | 0.1394 | 0.1415 | +12.2 % |
    | grid_import | all (1–48 h) | **0.1182** | 0.1341 | 0.1435 | **+11.9 %** |

    (kWh/h; mean actual 0.925 for export across 11 meters, 0.348 for import across 15.)
    The model beats the week-ago baseline on **23 of 26 device-targets**. Skill is highest
    where there is signal to remove — `...AAA3C` export +64 %, `...A0F18` +58 %,
    `...B17D0` +52 % — and the one clear loss is `...BC3F0` import (−52 %), a
    consumption-only meter whose naive error is already 0.0155 kWh/h, i.e. a series so flat
    that any model adds noise. Three device-targets are unscoreable on skill because their
    naive MAE is under 0.01 kWh/h.

12. **The shape of the error is not what the export case usually looks like.** For export
    the 24 h naive is a better baseline than the 168 h one at every range, and the model's
    hardest range is 9–24 h, not 25–48 h. Both follow from the cohort: these are small
    residential PV meters whose output is driven by the weather of the day, so yesterday
    resembles today more than last week does, and the 9–24 h range is where a forecast has
    to cover a whole solar day it cannot yet see any of.

### Which features are useful, per target

13. **`grid_export` — 24 of 51 features kept.** The useful ones, by how much shuffling them
    costs on the holdout: `theoretical_prod` **+73.8 %**, `global_tilted_irradiance`
    +48.2 %, `shortwave_radiation` +42.8 %, `grid_export_mean_same_hour_7d` +41.1 %,
    `gti_day_sum` +15.0 %, `ghi_ramp` +13.3 %, `grid_export_same_hour_1d` +13.0 %,
    `grid_export_median_same_hour_7d` +6.9 %, `gti_roll_3h` +6.5 %, `solar_elevation`
    +6.1 %, `grid_export_same_hour_2d` +5.3 %, `hours_to_sunset` +4.7 %,
    `hours_since_sunrise` +4.6 %, `hour_cos` +4.2 %, `cloud_cover` +4.2 %, `hour_sin`
    +2.4 %, `grid_export_same_hour_14d` +2.1 %, `clearsky_index` +2.0 %,
    `grid_export_same_hour_21d` +1.7 %, `grid_export_same_hour_3d` +1.4 %,
    `grid_export_same_hour_28d` +1.4 %, `effective_solar_pv` +1.0 %. Two more —
    `doy_sin` and `doy_cos` — are neutral on permutation but carry at least 1 % of gain, so
    the rule keeps them. By family, weather takes **75.9 % of the gain** and 224 points of
    ΔMAE; target history takes 19.7 % of gain but 74 points of ΔMAE; calendar takes 4.4 %
    of gain and 6 points of ΔMAE, almost all of it from the cyclic hour.

14. **`grid_import` — 28 of 51 features kept**, and the ordering is inverted: target
    history dominates. `grid_import_median_same_hour_7d` **+102.4 %**, `solar_elevation`
    +96.0 %, `grid_import_same_hour_1d` +75.1 %, `grid_import_mean_same_hour_7d` +46.7 %,
    `effective_solar_pv` +29.8 %, `theoretical_prod` +24.9 %, `ghi_ramp` +17.4 %,
    `grid_import_same_hour_2d` +13.9 %, `shortwave_radiation` +12.6 %,
    `grid_import_same_hour_3d` +8.5 %, `grid_import_same_hour_7d` +2.4 %, `hours_to_sunset`
    +2.1 %, `temp_lag_24h` +2.0 %, `hour_sin` +1.3 %, `gti_roll_3h` +1.3 %, `gti_day_sum`
    +1.2 %, `clearsky_index` +1.1 %. Eleven more are neutral on permutation but hold at
    least 1 % of gain and are kept on that: `global_tilted_irradiance`,
    `hours_since_sunrise`, `heating_degree`, `temp_range_day`, `temp_roll_24h`,
    `day_length_h`, `hour_cos`, `day_of_week`, `doy_sin`, `doy_cos`,
    `grid_import_diff_7d`. History carries **249 points of ΔMAE and 38.9 % of gain**,
    weather 192 points and 42.6 %, calendar 3 points and 11.8 %.

15. **Three results are worth stopping on.**
    * **`theoretical_prod` is the single most valuable export feature**, ahead of the two
      raw irradiance columns it is built from. Multiplying tilted irradiance by
      `cos(zenith)` turns "energy arriving somewhere" into "energy arriving on a panel at
      this sun angle", and the model pays 74 % more MAE without it.
    * **The origin-anchored rolling block is dead weight here.** `roll_24h_mean`,
      `roll_24h_std`, `roll_24h_max`, `value_at_origin`, `prev_day_total` and
      `zero_share_7d` all land within ±0.2 % on both targets, and `value_at_origin` earns
      **exactly zero gain** for export. The same-hour echoes already say what these say,
      and they say it at the right hour of the day rather than at whatever hour the origin
      happens to fall on.
    * **`horizon` earns nothing** — +0.00 % ΔMAE and 0.00 % gain for export, +0.01 % and
      0.04 % for import — and is dropped from both sets. One booster really does serve
      every horizon here, because every history feature is already anchored to the target
      hour and to its own admissible lag; the model has no use for being told how far ahead
      it is looking.

16. **What was dropped, and what it cost: essentially nothing.** Retraining all 26 models
    on the selected sets alone and scoring the same origins and the same hours:

    | target | features | MAE full pool | MAE selected | change |
    |---|---|---|---|---|
    | grid_export | 51 → 24 | 0.3116 | 0.3121 | **−0.16 %** |
    | grid_import | 51 → 28 | 0.1182 | **0.1169** | **+1.10 %** |

    Export loses a sixth of a per cent, all of it in the 1–8 h range (0.1884 → 0.2001,
    −6.2 %); it gains slightly at 9–24 h and 25–48 h. Import improves at every range past
    8 h. Per device-target the smaller model is at least as good on 13 of 26, and the wins
    and losses are small and two-sided, which is what a genuinely redundant half of a pool
    looks like. Halving the feature count for a rounding error is a good trade.

17. **The known duplication is real and visible.** The ramp test prints it: for `h = 36`
    and `h = 48` the implied lag of `same_hour_1d` is **48 h**, exactly that of
    `same_hour_2d`, and `diff_1d` is **identically 0.0** across the whole 25–48 h range — a
    feature that cannot split, and a duplicate column that dilutes `feature_fraction`
    sampling. The verdict confirms it from the other side: `diff_1d` scores +0.31 % for
    export and +0.05 % for import and is dropped for both. A 72 h fallback for
    `same_hour_1d` beyond 48 h would recover the slot at no cost.

### What is fragile about these numbers

18. **The early stopping is not starving the models — but the round budget is worth more
    than it looks.** The check at the end of 5.2 fits four device-targets three ways, and
    the first half of the answer is unambiguous: the small tree counts are the validation
    optimum, not an artefact. On `...B17D0` export the validation RMSE is **1.4580 at 30
    rounds, 1.4646 at 100 and 1.4610 at 300**, flat from about 50 trees on, and the run
    stops at 49. On `...C9968` export it *rises* monotonically (0.5297 → 0.5406 → 0.5471)
    and the run stops at 24. Both Tweedie models drift the same way (`...89CF4` 7.8087 →
    7.8166 → 7.8180, `...89ED4` 2.5644 → 2.5679 → 2.5711). Nothing is being cut short: the
    validation split asks for exactly these tree counts.

    The second half is less comfortable. **The validation split and the holdout disagree,
    and by a lot.** Forcing 300 rounds with no stopping moves the holdout MAE by
    **−1.6 %** (`...B17D0` export), **−25.3 %** (`...C9968` export), **−24.5 %**
    (`...89ED4` import) and **+9.6 %** (`...89CF4` import) — better on three of the four,
    worse on one, and the validation curve gives no hint of the sign. The slower fit
    (`learning_rate = 0.05`, up to 500 rounds, patience 50) is the steadier alternative:
    109 and 49 trees on the two exporters, 46 and 16 on the importers, and holdout changes
    of only **−2.3 %, +2.0 %, +1.8 % and −8.4 %**.

    Two things follow. The feature verdict of 5.3 is measured at each model's own
    validation optimum, which is the defensible place to measure it, and 5.4 compares the
    full pool against the selected set under the identical setting on both sides, so that
    comparison is sound. But the **absolute** MAEs of 5.2 carry a per-device uncertainty of
    up to a quarter from the round budget alone. They should be read as "the model clears
    the naive baselines by this much", not as tuned accuracy — and tuning the round budget
    per device, on something better than a single trailing validation split, is worth more
    here than any further feature work.

19. **The weather is an archive, not a forecast.** Every weather feature is read at the
    target hour from the Open-Meteo history, which is the same as assuming a perfect
    weather forecast. Since weather takes three quarters of the export model's gain, the
    export skill of +41.9 % is an upper bound; a real ICON-D2 forecast at 24–48 h would
    move it down by an amount this notebook cannot measure.

20. **Seven origins per device is a signal, not a verdict.** Each device-target is scored on
    336 hours from one week of its own history, and those weeks are not the same week
    across devices. A feature that only matters in winter cannot show up in a holdout that
    lands in June or September, which is the honest reading of `is_holiday` and `is_bridge`
    scoring exactly zero: the holdout windows contain no holiday, so the conclusion is "not
    measurable here", not "useless".

### What we do next

* **Re-run this notebook once notebook 01 includes the `mySET` history.** The five
  short-span meters would go from four months to a full record, which changes their
  training matrices and their own holdout weeks alike. The feature verdict and every
  holdout number in section 5 have to be redone on the longer series, and that is the
  first thing to do after the update — before the pooled model and before the forecast
  weather, because both of those would otherwise be decided on spans we know are about
  to change.
* **A pooled model across devices.** Eight of the fifteen meters have under a year of
  history in this extract and five have only four months — the five still waiting for
  their `mySET` record; the per-device models trained on those stop after a handful of
  trees. A single model over all fifteen, with the PV flag and a per-device
  scale as regime features, would let the short-history meters borrow the shape of the long
  ones — and it is the only setting in which device-level constants can earn anything.
* **Real weather forecasts instead of the archive.** Re-score the same holdout with the
  ICON-D2 forecast that was actually available at each origin. That turns finding 19 from a
  caveat into a number, and it may reorder the weather block: a feature that is strong on
  perfect weather and weak on forecast weather is not worth its place.
* **A seasonal check of the verdict.** Repeat section 5 with the holdout moved to winter
  and to spring. Heating degree-days, `temp_range_day` and the holiday flags all have their
  season, and a feature set selected on a September week is selected on the easiest weather
  of the year.
* **Three questions this run raised.** Should partial hours be *weighted* rather than
  scaled, since an hour rebuilt from one quarter carries four times the variance of a full
  one? Should the near-flat series (`...C9968` import, `...B3594` import, `...BC3F0`
  import) be modelled at all, or reported as "no flow" and skipped? And should a device be
  re-assessed after a long outage — the internal gaps have a heavy tail, 99th percentile
  590 h, so `...8AA78` reaches its holdout with only 62.5 % of its own hours present.